# FEF Inactivation — exploring eye movements and neurons in Python

Welcome! This notebook walks you through a real monkey neurophysiology dataset:
eye movements and single-neuron recordings from the **frontal eye field (FEF)**,
recorded before, during and after the FEF was temporarily switched off with a drug.

**You do not need to know much Python to work through this.** Every cell is
commented, and the code deliberately uses simple, explicit `for` loops rather than
clever one-liners, so you can read it top to bottom and follow what happens.

This notebook is a **project**, not a tour. Sections 1 to 11 build the tools;
section 12 is the project you actually carry out and write up.

**The project goal: investigate how reward value shapes the effect of FEF
inactivation.** Does silencing the FEF damage saccades to a valuable object as much
as saccades to a worthless one?

Along the way you will:

1. Load a session and understand what is inside it
2. Align the eye traces to the moment the monkey was told to move
3. **Detect each saccade** (fast eye movement) and measure it
4. Measure where the eye landed and how long it stayed on the object
5. Look at the recorded neuron as a raster and a PSTH
6. Learn to tell a real difference from noise
7. Carry out the project in four steps

**How to run a cell:** click on it and press `Shift + Enter`. Run the cells in
order from the top the first time through — later cells depend on earlier ones.

---
# 1. What is this dataset?

## The big question

The **frontal eye field (FEF)** is a patch of cortex that helps control where you
look. If it really drives eye movements, then switching it off should change how
the monkey looks at things — and it should do so **only for one side of space**,
because the left FEF controls looking to the right and vice versa.

To test this, the experimenter injected a small amount of **muscimol** into the FEF.
Muscimol temporarily silences neurons in a small area; the effect builds up over
tens of minutes and wears off later. So a single recording session naturally splits
into trials **before**, **during**, and **after** the injection.

## The task, one trial at a time

Each trial follows the same script:

| Step | What the monkey sees | Event code |
|---|---|---|
| 1 | A dot appears in the middle. The monkey must look at it and hold still. | `SHOWFIXCD` (2), `FIXSTARTCD` (3) |
| 2 | A **fractal picture** appears off to one side (left or right). The monkey must keep staring at the centre dot — not look at it yet. | `TGTCD` (6) |
| 3 | About **467 ms later**, the centre dot disappears. This is the **GO cue** — now the monkey may look at the fractal. | `FIXOFF` (5) |
| 4 | The monkey makes a fast eye movement (a **saccade**) to the fractal and gets juice. | `RWDCD` (10) |

The **reaction time (RT)** is how long after the GO cue the saccade started.
That is the main behavioural measure in this notebook.

The fractal was one of two kinds: a **good object** (large reward) or a **bad
object** (small reward). So each trial has both a *direction* (left or right) and a
*value* (good or bad).

## Why we have to detect the saccades ourselves

You might expect the file to just tell us the reaction time. It does not. There is
an event code that *could* mark the saccade (`SAC_OCCURRED`), but it was only
recorded on about **11% of trials**. So the reaction time has to be measured from
the **eye position trace** itself. Section 7 is where we do that, and it is the
heart of this notebook.

## The three sessions

| Short name | Task | Trials | Target positions | Injection phases |
|---|---|---|---|---|
| `Adams102325_FRAC` | Fractal object directed saccade | 1740 | 20° up-left / down-right | before / during / after |
| `Adams110725_FRAC` | Fractal object directed saccade | 1337 | 15° left / right | before / during / after |
| `Adams110725_OneDR` | One-direction-rewarded | 470 | 15° left / right | before / during only |

Start with `Adams102325_FRAC`. Later, change one line and re-run everything on
another session — a good habit, and a good test of whether your analysis was
accidentally tuned to one session.

## One important caveat, before you compute any percentage

**Only the trials the monkey got right are stored in these files.** Trials where he
broke fixation, never fixated, or failed the saccade were counted elsewhere and
then dropped. For the 110725 session there were 2137 trials in total but only 1337
correct ones — so roughly 800 trials are simply missing from what you can see.

This matters most for the *anticipation rate*. The trials where the monkey jumped
the gun hardest are exactly the ones that became fixation breaks and got thrown
away. So any anticipation rate you measure here is an **underestimate**, and a
biased one. Keep that in mind rather than reporting the number as if it were
the truth.

---
# 2. Setup: packages and data

## The packages

We only use four, and all of them are already installed on Google Colab:

- **numpy** — arrays of numbers, and the maths to go with them. Always imported as `np`.
- **scipy** — extra scientific tools. We use exactly one function from it (a smoother).
- **matplotlib** — all the plotting. We use `matplotlib.pyplot`, always imported as `plt`.
- **pandas** — only to print tidy tables at the end. Imported as `pd`.

In [ ]:
# "import X as Y" means: load the toolbox X, and refer to it by the short name Y.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import savgol_filter   # the one scipy function we need

# Make the figures a comfortable size and reasonably readable.
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.spines.top"] = False     # drop the box around plots;
plt.rcParams["axes.spines.right"] = False   # it is just visual clutter

print("Packages loaded. numpy version:", np.__version__)

## Getting the data file

The data has been packed into a small `.npz` file (a `.npz` is simply a box holding
several numpy arrays). Each session is 6–24 MB, so downloading takes a few seconds.

**Choose your session here.** This is the only line you need to change to analyse a
different recording.

In [ ]:
# ============================================================================
# CHOOSE YOUR SESSION -- change this one line to analyse a different recording
# ============================================================================
SESSION = "Adams102325_FRAC"      # or "Adams110725_FRAC" or "Adams110725_OneDR"


# ---------------------------------------------------------------------------
# Where to get the file from.
#
# On Google Colab, leave USE_LOCAL_FILE as False. The data downloads itself.
#
# If you are running this notebook on your own computer and already have the
# .npz files, set USE_LOCAL_FILE = True and point LOCAL_FOLDER at them.
# ---------------------------------------------------------------------------
USE_LOCAL_FILE = False
LOCAL_FOLDER = "."

# Where the data files live. These are already filled in -- you should not need
# to change them.
RELEASE = "https://github.com/xuefeiyu2015/fef-inactivation-notebook/releases/download/data-v1"

DATA_URLS = {
    "Adams102325_FRAC": RELEASE + "/Adams102325_FRAC.npz",
    "Adams110725_FRAC": RELEASE + "/Adams110725_FRAC.npz",
    "Adams110725_OneDR": RELEASE + "/Adams110725_OneDR.npz",
}

In [ ]:
import os
import urllib.request

file_name = SESSION + ".npz"

if USE_LOCAL_FILE:
    data_path = os.path.join(LOCAL_FOLDER, file_name)

else:
    data_path = file_name

    if os.path.exists(data_path):
        print("Already downloaded:", data_path)
    else:
        print("Downloading", file_name, "...")
        urllib.request.urlretrieve(DATA_URLS[SESSION], data_path)
        print("Download finished.")

# Check the file really arrived before going any further.
if not os.path.exists(data_path):
    raise FileNotFoundError(
        "Could not find " + data_path + ".\n"
        "If you are on Colab, check that you are connected to the internet and\n"
        "run this cell again."
    )

size_mb = os.path.getsize(data_path) / 1e6
print("Data file ready:", data_path, "(%.1f MB)" % size_mb)

---
# 3. Loading the data and looking inside

`np.load` opens the `.npz` box. What comes back behaves like a **dictionary**: a
collection of named items, where you get an item by writing `box["name"]`.

Let us open it and list everything inside.

In [ ]:
session = np.load(data_path, allow_pickle=False)

print("This file contains", len(session.files), "items:\n")
for name in session.files:
    item = session[name]
    print("  %-22s shape %-16s type %s" % (name, str(item.shape), item.dtype))

### What those shapes mean

Look at the shapes above. Most items have a first number equal to the **number of
trials** — 1740 for the 102325 session. That is the key idea:

> **Row `i` of almost every array is trial `i`.**

So `target_x[5]` is the target position on trial 5, and `eye_x[5]` is the eye trace
on trial 5. Everything lines up, which is what makes it possible to ask questions
like "was the reaction time slower on the trials where the target was on the right?"

### A word about `nan`

You will see `nan` a lot. It stands for **"not a number"**, and it is how we write
*missing data*. It is not zero — zero is a real measurement, `nan` means we do not
know.

Two things to remember about `nan`:

1. Ordinary maths on `nan` gives `nan`. So `np.mean([1, 2, np.nan])` is `nan`.
   To ignore the missing values, use the `nan`-aware versions:
   **`np.nanmean`, `np.nanmedian`, `np.nansum`, `np.nanmax`**.
2. `nan` is not equal to anything, *including itself*. So `x == np.nan` is always
   False and never works. To test for it, use **`np.isnan(x)`**.

In [ ]:
# A quick demonstration, because this trips up nearly everyone at first.
values = np.array([1.0, 2.0, np.nan, 4.0])

print("np.mean    ->", np.mean(values), "   <- one missing value poisons the whole answer")
print("np.nanmean ->", np.nanmean(values), "   <- this is what you usually want")
print()
print("values == np.nan ->", values == np.nan, "  <- never use this, it is always False")
print("np.isnan(values) ->", np.isnan(values), "  <- use this instead")

---
# 4. The main variables

Here is everything in the file and what it means. **The single most important fact:**

> **Every time in this dataset is in milliseconds, counted from the start of that
> trial.** Event times, eye samples and spike times all use that same clock.

That is what makes the whole analysis possible: to ask "how long after the GO cue
did the spike happen?", you just *subtract* the GO cue time from the spike time.

### The eye traces

| Variable | Shape | Meaning |
|---|---|---|
| `eye_x` | trials × samples | Horizontal eye position, in **degrees of visual angle**. Positive = right. |
| `eye_y` | trials × samples | Vertical eye position, in degrees. Positive = up. |
| `eye_n_samples` | trials | How many real samples that trial has. |
| `eye_bin_width` | one number | Milliseconds between samples. It is **1**, i.e. 1000 samples per second. |

Trials have different lengths (roughly 3600–7300 samples). To store them in one
rectangular array, the short ones were padded at the end with `nan`. So the real
data of trial `i` is `eye_x[i, :eye_n_samples[i]]` — "row i, up to its real length".

Because the samples are 1 ms apart and start at the beginning of the trial,
**sample number = time in milliseconds**. Sample 500 happened 500 ms into the trial.

### The events

| Variable | Shape | Meaning |
|---|---|---|
| `event_codes` | trials × 18 | *What* happened, as numeric codes, in the order it happened. |
| `event_times` | trials × 18 | *When* each of those happened (ms from trial start). |

The two line up: if `event_codes[i, 3]` is `6`, then `event_times[i, 3]` is when
code `6` happened on trial `i`. Trials with fewer than 18 events are padded with `nan`.

### The target

| Variable | Shape | Meaning |
|---|---|---|
| `target_x`, `target_y` | trials | Where the fractal appeared, in degrees. |
| `target_angle` | trials | Same position as an angle. **0 = up, counting counter-clockwise**, so 90 = left, 180 = down, 270 = right. |
| `target_ecc` | trials | Distance from the centre, in degrees (15 or 20 here). |

⚠️ **Two traps worth knowing about**, both already handled for you:

- **Never test the angle with `==`.** The stored values are numbers like
  `45.00000002`, so `target_angle == 45` matches *nothing at all* and silently
  gives you an empty group. Use `target_x < 0` and `target_x > 0` instead — the
  sign of a number is safe to test.
- **Do not use the hemifield event codes to work out direction.** Each session
  used different codes for this (102/105 in one session, 80/81 in another), so code
  written around one session's numbers quietly produces empty groups on the next.
  `target_x` exists in every session and means the same thing every time.

### The injection

`injection_condition` has one value per trial:

| Value | Meaning |
|---|---|
| `0` | **before** the injection |
| `nan` | **during** the injection |
| `1` | **after** the injection |

Note that "during" is written as `nan`, so you must find those trials with
`np.isnan(injection_condition)` — not with `== np.nan`, which never works.

### The neuron

| Variable | Shape | Meaning |
|---|---|---|
| `spike_times` | trials × max spikes | The time of every spike, in ms from trial start, padded with `nan`. |

One neuron was recorded. Row `i` holds that trial's spike times; a trial with 35
spikes has 35 numbers followed by `nan` padding.

Let us now pull everything out of the box and into plain variables.

In [ ]:
# Pull each item out of the box and give it a short name we can use from now on.
eye_x = session["eye_x"]
eye_y = session["eye_y"]
eye_n_samples = session["eye_n_samples"]
eye_bin_width = float(session["eye_bin_width"])

event_codes = session["event_codes"]
event_times = session["event_times"]

target_x = session["target_x"]
target_y = session["target_y"]
target_angle = session["target_angle"]
target_ecc = session["target_ecc"]

injection_condition = session["injection_condition"]
spike_times = session["spike_times"]

# How many trials are there? The number of rows of any per-trial array.
n_trials = len(target_x)

print("Session:      ", str(session["session_name"]))
print("Task:         ", str(session["task_type"]), "(code %d)" % int(session["task_code"]))
print("Trials:       ", n_trials)
print("Eye sampling: ", "one sample every %.0f ms" % eye_bin_width)
print("Total spikes: ", int(np.sum(~np.isnan(spike_times))))

Let us look at one single trial, to make the shapes concrete.

In [ ]:
trial = 0     # Python counts from 0, so this is the FIRST trial

n_samples_here = eye_n_samples[trial]

print("Trial", trial)
print("  eye trace length:   %d samples = %d ms" % (n_samples_here, n_samples_here * eye_bin_width))
print("  target position:    x = %.1f deg, y = %.1f deg" % (target_x[trial], target_y[trial]))
print("  target angle:       %.2f deg   <- note it is not exactly a round number!" % target_angle[trial])
print("  eccentricity:       %.0f deg" % target_ecc[trial])
print()

# The eye position during the first 10 ms of the trial.
print("  first 10 eye_x samples:", np.round(eye_x[trial, :10], 2))
print()

# The spikes on this trial: drop the nan padding to see the real ones.
this_trial_spikes = spike_times[trial]
this_trial_spikes = this_trial_spikes[~np.isnan(this_trial_spikes)]
print("  number of spikes:   ", len(this_trial_spikes))
print("  first 5 spike times:", np.round(this_trial_spikes[:5], 1), "ms from trial start")

Notice the `~np.isnan(...)` in the cell above. That is the standard way to say
**"keep only the parts that are not missing"**:

- `np.isnan(x)` gives True where the value is missing
- `~` means **not**, so `~np.isnan(x)` is True where the value is real
- `x[~np.isnan(x)]` keeps only the real values

You will use this pattern constantly. It is worth reading that line until it feels
obvious.

---
# 5. Event codes: when did each thing happen?

The events are stored as numeric codes. Here are the ones that matter for us:

| Code | Name | Meaning |
|---|---|---|
| 2 | `SHOWFIXCD` | The central fixation dot appeared |
| 3 | `FIXSTARTCD` | The monkey started fixating it |
| 6 | `TGTCD` | **The fractal target appeared** |
| 5 | `FIXOFF` | **The GO cue** — fixation dot off, "you may look now" |
| 7 | `SAC_OCCURRED` | A saccade was detected online (only ~11% of trials — unreliable) |
| 10 | `RWDCD` | Reward delivered |
| 12 | `CORRECTCD` | Trial marked correct |
| 90 / 91 | `GOODOBJ` / `BADOBJ` | The fractal was the high-value / low-value object |

Notice that `FIXOFF` is code **5** and `TGTCD` is code **6** — the GO cue has the
*smaller* number even though it happens *later*. The codes are just labels, not an
order, so always look up the code by name rather than assuming.

To find *when* a code happened on each trial, we search that trial's row of
`event_codes` for the code, then read the matching entry of `event_times`.

In [ ]:
def find_event_time(event_codes, event_times, wanted_code):
    """Find when one event code happened, on every trial.

    Returns one number per trial: the time in ms from that trial's start.
    Trials where the code never occurred get nan.
    """
    n_trials = event_codes.shape[0]

    # Start with "missing everywhere", then fill in the trials where we find it.
    result = np.full(n_trials, np.nan)

    for i in range(n_trials):
        # Which positions in this trial's row hold the code we want?
        positions = np.where(event_codes[i] == wanted_code)[0]

        if len(positions) > 0:
            first_position = positions[0]      # if it happened twice, take the first
            result[i] = event_times[i, first_position]

    return result


# The two markers we care about most.
target_on_time = find_event_time(event_codes, event_times, 6)   # TGTCD
go_cue_time = find_event_time(event_codes, event_times, 5)      # FIXOFF

print("Target onset found on %d of %d trials" % (np.sum(~np.isnan(target_on_time)), n_trials))
print("GO cue found on       %d of %d trials" % (np.sum(~np.isnan(go_cue_time)), n_trials))

# How long did the monkey have to wait between seeing the target and being allowed
# to look at it?
delay = go_cue_time - target_on_time
print()
print("Delay from target onset to GO cue: %.0f +/- %.0f ms" % (np.nanmean(delay), np.nanstd(delay)))

**That delay is the key to a puzzle you will meet in section 8.** The wait is
almost always the same length — about 467 ms, varying by only ±9 ms. The monkey
did hundreds of these trials, so he learned the timing and started moving his eyes
*before* the GO cue actually appeared. That is why you will see some **negative
reaction times**, and they are real behaviour, not a bug.

Now let us check how often the online saccade marker was recorded — the reason we
have to detect saccades ourselves.

In [ ]:
def count_trials_with_code(event_codes, wanted_code):
    """Count how many trials contain a given event code at least once."""
    count = 0
    for i in range(event_codes.shape[0]):
        if np.any(event_codes[i] == wanted_code):
            count = count + 1
    return count


n_with_marker = count_trials_with_code(event_codes, 7)   # SAC_OCCURRED
print("SAC_OCCURRED was recorded on %d of %d trials (%.0f%%)"
      % (n_with_marker, n_trials, 100 * n_with_marker / n_trials))
print()
print("-> far too few to use. We will detect the saccades from the eye trace instead.")

## Sorting the trials into conditions

Now we build **masks**. A mask is an array of True/False, one entry per trial, that
says which trials belong to a group. You then use it to pick out those trials:
`reaction_time[is_left_saccade]` gives the reaction times of just the leftward trials.

In [ ]:
# --- Which way did the monkey have to look? ---
# Take this from the sign of the target's x position -- NOT from the hemifield
# event codes, which are different numbers in different sessions.
is_left_saccade = target_x < 0
is_right_saccade = target_x > 0

# --- Was the fractal the high-value or the low-value object? ---
def make_code_mask(event_codes, wanted_code):
    """True on every trial that contains the given event code."""
    n_trials = event_codes.shape[0]
    mask = np.zeros(n_trials, dtype=bool)      # start with all False
    for i in range(n_trials):
        if np.any(event_codes[i] == wanted_code):
            mask[i] = True
    return mask

is_good_object = make_code_mask(event_codes, 90)   # GOODOBJ, large reward
is_bad_object = make_code_mask(event_codes, 91)    # BADOBJ, small reward

print("Saccade direction:  %4d leftward, %4d rightward" % (np.sum(is_left_saccade), np.sum(is_right_saccade)))
print("Object value:       %4d good,     %4d bad" % (np.sum(is_good_object), np.sum(is_bad_object)))
print("Target eccentricity: %.0f deg" % np.nanmedian(target_ecc))

### The injection phases

Rather than assuming every session is "before / during / after", we **read the
phases out of the data**. Sessions differ, and code that assumes one layout breaks
silently on the next one — for example the OneDR session has no "after" phase at
all, because the recording ended while the drug was still active.

In [ ]:
def compute_injection_phases(injection_condition):
    """Work out which injection phases this session actually has.

    Returns a list of (name, mask) pairs, in the order before -> during -> after.
    """
    phases = []

    # "before" and "after" are ordinary numbers, "during" is nan, so it needs
    # np.isnan rather than a comparison.
    if np.any(injection_condition == 0):
        phases.append(("before", injection_condition == 0))

    if np.any(np.isnan(injection_condition)):
        phases.append(("during", np.isnan(injection_condition)))

    if np.any(injection_condition == 1):
        phases.append(("after", injection_condition == 1))

    # A session with a SECOND injection would carry the value 2. None of the three
    # sessions here does, but we handle it so nothing gets silently dropped.
    if np.any(injection_condition == 2):
        phases.append(("after 2nd injection", injection_condition == 2))

    return phases


injection_phases = compute_injection_phases(injection_condition)

print("This session has %d injection phases:" % len(injection_phases))
for name, mask in injection_phases:
    print("   %-20s %4d trials" % (name, np.sum(mask)))

# Safety check: every trial should belong to exactly one phase.
total_in_phases = 0
for name, mask in injection_phases:
    total_in_phases = total_in_phases + np.sum(mask)
print()
print("Trials covered by a phase: %d of %d" % (total_in_phases, n_trials))

---
# 6. Aligning the eye traces to the GO cue

## The problem

Every trial is recorded separately and has its own length, and the GO cue happens at
a **different moment in every trial** — around 2200 ms into the trial, but never at
exactly the same time. So sample 2200 means something different on every trial, and
you cannot compare or average the traces as they are.

## The fix

Cut a window around the GO cue on each trial, and place all those windows on **one
shared time axis where 0 means "the GO cue"**. After that, column `k` of the result
is always the same time relative to the GO cue, on every trial. Now the trials can
be compared, averaged and plotted together.

We will keep 300 ms before the GO cue and 600 ms after it.

## Why interpolation and not just cutting

The GO cue happened at, say, 2191.95 ms — not on a whole millisecond. But the eye
was only sampled at whole milliseconds (2191, 2192, ...). So to know where the eye
was at exactly 2191.95 ms, we read *between* two samples. That is **interpolation**,
and `np.interp` does it: given the known sample times and values, it estimates the
value at any time you ask for.

`np.interp` needs three things: the times you *want*, the times you *have*, and the
values you have. Anything outside the recorded range comes back as `nan`.

In [ ]:
def align_eye_traces(eye_x, eye_y, eye_n_samples, eye_bin_width,
                     align_time, ms_before, ms_after):
    """Cut a window out of every trial's eye trace, centred on a marker.

    align_time gives the marker time for each trial (ms from trial start).
    Returns aligned_x, aligned_y (trials x samples) and the shared time axis,
    on which 0 is the marker. Trials with no marker come back as all nan.
    """
    n_trials = len(align_time)

    # The shared time axis: -300, -299, ..., 0, ..., 599, 600 (in ms).
    time_axis = np.arange(-ms_before, ms_after + 1) * eye_bin_width

    # Start with everything missing, then fill trial by trial.
    aligned_x = np.full((n_trials, len(time_axis)), np.nan)
    aligned_y = np.full((n_trials, len(time_axis)), np.nan)

    for i in range(n_trials):
        if np.isnan(align_time[i]):
            continue           # no marker on this trial -> leave the row as nan

        # This trial's real data, with the nan padding removed.
        n_here = eye_n_samples[i]
        trial_x = eye_x[i, :n_here]
        trial_y = eye_y[i, :n_here]

        # When each of those samples was recorded, in ms from the trial start.
        # Samples are 1 ms apart, so this is simply 0, 1, 2, 3, ...
        sample_time = np.arange(n_here) * eye_bin_width

        # The moments we WANT to know about, on the trial's own clock.
        wanted_time = align_time[i] + time_axis

        # Read the trace at those moments. left/right say what to use when the
        # window reaches outside the recording: nan, i.e. "we do not know".
        aligned_x[i] = np.interp(wanted_time, sample_time, trial_x, left=np.nan, right=np.nan)
        aligned_y[i] = np.interp(wanted_time, sample_time, trial_y, left=np.nan, right=np.nan)

    return aligned_x, aligned_y, time_axis


MS_BEFORE = 300     # how much eye trace to keep before the GO cue
MS_AFTER = 600      # ...and after it

aligned_x, aligned_y, eye_time = align_eye_traces(
    eye_x, eye_y, eye_n_samples, eye_bin_width, go_cue_time, MS_BEFORE, MS_AFTER)

print("Aligned eye traces shape:", aligned_x.shape, " (trials x samples)")
print("Time axis runs from %.0f to %.0f ms, with 0 = the GO cue" % (eye_time[0], eye_time[-1]))

# A few trials are short enough that the window runs off the end of the recording.
# Those get some nan, which is honest -- we genuinely do not have that data.
n_incomplete = np.sum(np.any(np.isnan(aligned_x), axis=1))
print()
print("Trials with some missing samples in the window: %d of %d" % (n_incomplete, n_trials))

Let us look at what we have got. Below are the first 30 trials, split by which side
the target was on. Each thin line is one trial's horizontal eye position.

Before the GO cue (the dashed line at 0) the eye sits near the centre, at 0 degrees.
Shortly after, it jumps — up to positive values for rightward targets, down to
negative for leftward ones. **That jump is the saccade**, and measuring it is the
job of the next section.

In [ ]:
def plot_example_traces(time_axis, aligned_x, is_left, is_right, n_show=30):
    """Draw the horizontal eye position of a few trials, split by target side."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    groups = [("Leftward target", is_left, "tab:blue"),
              ("Rightward target", is_right, "tab:red")]

    for ax, (title, mask, colour) in zip(axes, groups):
        trials_to_show = np.where(mask)[0][:n_show]

        for i in trials_to_show:
            ax.plot(time_axis, aligned_x[i], color=colour, linewidth=0.7, alpha=0.6)

        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.text(5, ax.get_ylim()[1] * 0.9, "GO cue", fontsize=9)
        ax.set_title("%s  (%d trials shown)" % (title, len(trials_to_show)))
        ax.set_xlabel("Time from GO cue (ms)")

    axes[0].set_ylabel("Horizontal eye position (deg)")
    fig.suptitle("Raw aligned eye traces -- the saccade is the sudden jump", y=1.02)
    fig.tight_layout()
    plt.show()


plot_example_traces(eye_time, aligned_x, is_left_saccade, is_right_saccade)

---
# 7. Detecting the saccade

## The idea, in five steps

A saccade is a *fast* eye movement. So instead of looking at eye **position**, we
look at eye **speed** — how fast the eye is moving, in degrees per second. During
fixation the speed is near zero; during a saccade it shoots up to over 1000 deg/s.

1. **Smooth** the eye position a little, then turn it into **speed**.
2. Measure the **resting speed** in a quiet window before the GO cue.
3. A saccade is where speed rises **clearly above** that resting level and **stays**
   there for several samples in a row (one noisy sample is not enough).
4. Walk **backwards** from there to the moment the speed first crossed the
   threshold. That instant is the **saccade onset**, and its time is the **reaction time**.
5. Walk **forwards** past the peak to where the speed settles back down. That is the
   **saccade offset**.

## Step 1: position → speed

Two details matter here.

**Why smooth first?** The eye tracker is a little noisy. Speed is computed by
subtracting neighbouring samples, and subtracting neighbours *amplifies* noise
badly. Smoothing first keeps the speed trace readable. We use a Savitzky-Golay
filter, which smooths while preserving the sharp shape of a real saccade better
than a plain running average would.

**Handling missing data.** `savgol_filter` crashes if the trace contains any `nan`,
and some of our traces do at the edges. So the helper below smooths only the real
part of each trace and leaves the rest missing.

In [ ]:
def smooth_trace(trace, span=7, poly_order=2):
    """Smooth one trial's trace, coping with the nan padding at its edges.

    savgol_filter refuses to run on data containing nan, so we find the stretch of
    real data and smooth only that part.
    """
    smoothed = np.full(len(trace), np.nan)

    real_positions = np.where(~np.isnan(trace))[0]
    if len(real_positions) < span:
        return smoothed          # too little real data to smooth

    first = real_positions[0]
    last = real_positions[-1] + 1

    # If there is a gap of missing data in the MIDDLE, give up on this trial
    # rather than inventing values across the hole.
    if np.any(np.isnan(trace[first:last])):
        return smoothed

    smoothed[first:last] = savgol_filter(trace[first:last], span, poly_order)
    return smoothed


def compute_eye_speed(aligned_x, aligned_y, time_axis, smooth_span=7):
    """Smooth the eye position and convert it into speed in degrees per second.

    Returns the smoothed x and y, the speed, and the time axis the speed sits on.
    """
    n_trials = aligned_x.shape[0]

    smooth_x = np.full(aligned_x.shape, np.nan)
    smooth_y = np.full(aligned_y.shape, np.nan)

    for i in range(n_trials):
        smooth_x[i] = smooth_trace(aligned_x[i], smooth_span)
        smooth_y[i] = smooth_trace(aligned_y[i], smooth_span)

    # Time between two samples, in seconds (our samples are 1 ms apart).
    step_ms = np.median(np.diff(time_axis))
    step_seconds = step_ms / 1000.0

    # How far did the eye move between neighbouring samples, in x and in y?
    step_x = np.diff(smooth_x, axis=1)
    step_y = np.diff(smooth_y, axis=1)

    # Combine the two into a single distance with Pythagoras, then divide by the
    # time taken to get a speed. np.hypot(a, b) is sqrt(a*a + b*b).
    speed = np.hypot(step_x, step_y) / step_seconds

    # np.diff gives one fewer value than we started with, because each value
    # describes the gap BETWEEN two samples. So the speed sits at the midpoints.
    speed_time = (time_axis[:-1] + time_axis[1:]) / 2.0

    return smooth_x, smooth_y, speed, speed_time


smooth_x, smooth_y, eye_speed, speed_time = compute_eye_speed(aligned_x, aligned_y, eye_time)

print("Speed array shape:", eye_speed.shape)
print("Fastest speed seen anywhere: %.0f deg/s" % np.nanmax(eye_speed))

## Step 2: the settings

All the tuning lives in one place, so you can change one number and re-run. The
defaults below are the ones that were checked against the original MATLAB analysis.

In [ ]:
SETTINGS = {
    "smooth_span": 7,          # samples used by the smoother (about 7 ms)
    "baseline_window": (-200, -50),   # a quiet stretch, used to measure resting speed
    "search_window": (-50, 400),      # only look for a saccade onset in here
    "velocity_threshold": 40,  # deg/s ABOVE resting speed to count as moving
    "n_contiguous": 5,         # samples the speed must stay above, in a row
    "max_duration_ms": 200,    # a saccade cannot last longer than this
    "min_amplitude_deg": 3,    # anything smaller is drift, not a saccade
    "noise_speed": 2500,       # faster than this is a blink or a tracker glitch
    "offset_window_factor": 2,    # the offset is judged over a longer window...
    "offset_min_fraction": 0.5,   # ...of which only half needs to be below threshold
}

### Two settings worth understanding

**`noise_speed` (the blink guard).** Blinks and tracker glitches produce
impossibly fast "movements", so anything above this speed is thrown out. The trap
is that this number must sit **above the fastest real saccade**, or it quietly
deletes good data instead of bad. Real 20° saccades here peak at 1200–1500 deg/s,
and the original analysis found that setting this to 1500 threw away **a quarter of
the good trials**, while 2500 keeps 98.7% of them and still catches genuine blinks.
If you ever change the smoothing or use a recording with a different sampling rate,
check this number again.

**`search_window` starts at −50 ms, before the GO cue.** That is deliberate.
Remember the delay was almost always 467 ms — the monkey learned it and sometimes
started moving *before* the GO cue. If we only looked after 0, we would miss those
trials entirely and bias our reaction times.

## Step 3: finding the saccade

Two small helper functions do the searching. They are worth reading, because the
difference between them is the whole reason the detector works well.

In [ ]:
def find_first_run(flags, search_positions, n_needed):
    """Find the first place where flags is True n_needed times IN A ROW.

    Requiring a run, rather than a single sample, is what stops one noisy sample
    from being mistaken for a saccade. Returns None if there is no such run.
    """
    last_possible_start = len(flags) - n_needed

    for position in search_positions:
        if position < 0 or position > last_possible_start:
            continue
        if np.all(flags[position:position + n_needed]):
            return position

    return None


def find_first_majority(flags, search_positions, window_length, n_needed):
    """Find the first window of window_length samples containing at least
    n_needed True values. Returns the START of that window, or None.

    This is the relaxed cousin of find_first_run: that one asks "are they ALL
    true?", this one asks "are ENOUGH of them true?".
    """
    last_possible_start = len(flags) - window_length

    for position in search_positions:
        if position < 0 or position > last_possible_start:
            continue
        if np.sum(flags[position:position + window_length]) >= n_needed:
            return position

    return None

**Why two different rules?** The *start* of a saccade is clean: the speed rises
through the threshold and stays up. So `find_first_run` — all samples above — works
well there.

The *end* is messy. The speed undershoots, wobbles, and a small corrective movement
often pushes a sample or two back above the threshold. Demanding that *every* sample
be below the threshold makes the search walk straight past the real landing point
and report the saccade as longer than it was. So the offset uses the looser
`find_first_majority` rule: half of a 10-sample window below the threshold is
enough. In the original analysis this shortened the median duration from 44 ms to a
much more realistic 38 ms, without changing the onset at all.

One more subtlety: **the search for the offset starts after the speed PEAK**, not
right after the onset. On the way up, the speed is still climbing through the
threshold, so some samples are below it — a "half the window" rule could be
satisfied there and end the saccade before it began. A saccade's end must come
after its fastest moment, so that is where we start looking.

Now the detector itself.

In [ ]:
def detect_saccades(smooth_x, smooth_y, speed, speed_time, settings):
    """Find and measure the saccade on every trial.

    Returns a dictionary of arrays, one entry per trial, in trial order.
    Undetected trials are nan everywhere, so every array still lines up with
    the other per-trial variables.
    """
    n_trials = speed.shape[0]

    # Prepare the answers: everything missing until we fill it in.
    detected = np.zeros(n_trials, dtype=bool)
    reaction_time = np.full(n_trials, np.nan)
    saccade_end = np.full(n_trials, np.nan)
    amplitude = np.full(n_trials, np.nan)
    angle = np.full(n_trials, np.nan)
    peak_velocity = np.full(n_trials, np.nan)
    start_x = np.full(n_trials, np.nan)
    start_y = np.full(n_trials, np.nan)
    end_x = np.full(n_trials, np.nan)
    end_y = np.full(n_trials, np.nan)
    onset_index = np.full(n_trials, np.nan)
    offset_index = np.full(n_trials, np.nan)

    # Mark impossibly fast samples as noise (blinks, tracker glitches). They are
    # set to nan, so they can never trigger a detection.
    is_noise = speed > settings["noise_speed"]
    speed = np.where(is_noise, np.nan, speed)

    # Which samples fall in the baseline window, and which in the search window?
    # These are the same for every trial, so work them out once.
    base_lo, base_hi = settings["baseline_window"]
    in_baseline = (speed_time >= base_lo) & (speed_time <= base_hi)

    search_lo, search_hi = settings["search_window"]
    search_positions = np.where((speed_time >= search_lo) & (speed_time <= search_hi))[0]

    step_ms = np.median(np.diff(speed_time))
    max_duration_samples = int(round(settings["max_duration_ms"] / step_ms))

    # The offset rule: a window this long, this many of which must be below.
    offset_window = int(round(settings["n_contiguous"] * settings["offset_window_factor"]))
    offset_n_needed = max(1, int(round(offset_window * settings["offset_min_fraction"])))

    for i in range(n_trials):
        trial_speed = speed[i]

        # --- the resting speed on this trial, and the threshold from it ---
        baseline_samples = trial_speed[in_baseline]
        if np.all(np.isnan(baseline_samples)):
            continue                          # no usable baseline on this trial
        baseline = np.nanmean(baseline_samples)
        threshold = baseline + settings["velocity_threshold"]

        # Note these are kept separate on purpose: a nan (noise) sample is neither
        # above nor below, so a glitch can neither start nor end a saccade.
        is_above = trial_speed > threshold
        is_below = trial_speed < threshold

        # --- the onset ---
        trigger = find_first_run(is_above, search_positions, settings["n_contiguous"])
        if trigger is None:
            continue                          # no saccade found on this trial

        # The trigger is where we became SURE. The true onset is a little earlier,
        # where the speed first crossed the threshold on its way up.
        onset = trigger
        while onset > 0 and trial_speed[onset - 1] >= threshold:
            onset = onset - 1

        # --- the offset ---
        last_allowed = min(onset + max_duration_samples, len(trial_speed) - 1)

        stretch = trial_speed[onset:last_allowed + 1]
        if np.all(np.isnan(stretch)):
            continue
        peak = onset + int(np.nanargmax(stretch))     # where the speed peaked

        offset = find_first_majority(is_below, range(peak + 1, last_allowed + 1),
                                     offset_window, offset_n_needed)
        if offset is None:
            offset = last_allowed        # still moving at the limit: stop there

        # --- reject the trial if a blink landed inside the saccade ---
        if np.any(is_noise[i, onset:offset + 1]):
            continue

        # --- measure it ---
        # Speed sample k describes the gap between position samples k and k+1,
        # so the movement runs from position onset to position offset+1.
        from_x = smooth_x[i, onset]
        from_y = smooth_y[i, onset]
        to_x = smooth_x[i, offset + 1]
        to_y = smooth_y[i, offset + 1]

        if np.isnan(from_x) or np.isnan(to_x):
            continue

        this_amplitude = np.hypot(to_x - from_x, to_y - from_y)
        if this_amplitude < settings["min_amplitude_deg"]:
            continue                          # drift, not a saccade

        detected[i] = True
        reaction_time[i] = speed_time[onset]
        saccade_end[i] = speed_time[offset]
        amplitude[i] = this_amplitude
        peak_velocity[i] = np.nanmax(trial_speed[onset:offset + 1])
        start_x[i] = from_x
        start_y[i] = from_y
        end_x[i] = to_x
        end_y[i] = to_y
        onset_index[i] = onset
        offset_index[i] = offset

        # The direction the eye actually travelled, written in the SAME convention
        # as target_angle (0 = up, counter-clockwise). np.arctan2 gives the usual
        # maths convention (0 = right), so the +270 and the wrap line them up.
        direction = np.degrees(np.arctan2(to_y - from_y, to_x - from_x))
        angle[i] = np.mod(direction + 270, 360)

    return {
        "detected": detected,
        "reaction_time": reaction_time,
        "saccade_end": saccade_end,
        "duration": saccade_end - reaction_time,
        "amplitude": amplitude,
        "angle": angle,
        "peak_velocity": peak_velocity,
        "start_x": start_x,
        "start_y": start_y,
        "end_x": end_x,
        "end_y": end_y,
        "onset_index": onset_index,
        "offset_index": offset_index,
    }


saccades = detect_saccades(smooth_x, smooth_y, eye_speed, speed_time, SETTINGS)

n_detected = np.sum(saccades["detected"])
print("Saccade detected on %d of %d trials (%.1f%%)" % (n_detected, n_trials, 100 * n_detected / n_trials))

## Did it work? Look at the traces before you trust any number

This is the most important figure in the notebook. Numbers from a detector you have
not looked at are worthless. Each panel below is one trial:

- the **top** row shows eye position (x in blue, y in orange)
- the **bottom** row shows eye speed, with the trial's threshold as a dashed line
- the **green line** is where the detector put the saccade **onset**
- the **red line** is where it put the **offset**

The onset should land right where the position starts to move and the speed starts
to shoot up. The offset should land where the eye arrives and the speed collapses.

In [ ]:
def plot_detection_check(time_axis, speed_time, smooth_x, smooth_y, speed,
                         saccades, settings, trials_to_show):
    """Draw position and speed for a few trials, with the detected saccade marked.

    This function only draws what it is handed -- it does no detecting of its own,
    so what you see is exactly what the detector reported.
    """
    n_show = len(trials_to_show)
    fig, axes = plt.subplots(2, n_show, figsize=(3.4 * n_show, 6), sharex=True)

    for column, i in enumerate(trials_to_show):
        ax_pos = axes[0, column]
        ax_speed = axes[1, column]

        ax_pos.plot(time_axis, smooth_x[i], color="tab:blue", label="eye x")
        ax_pos.plot(time_axis, smooth_y[i], color="tab:orange", label="eye y")
        ax_speed.plot(speed_time, speed[i], color="black", linewidth=1)

        # The onset and offset the detector found on this trial.
        if saccades["detected"][i]:
            onset_ms = saccades["reaction_time"][i]
            offset_ms = saccades["saccade_end"][i]
            for ax in (ax_pos, ax_speed):
                ax.axvline(onset_ms, color="tab:green", linewidth=1.5)
                ax.axvline(offset_ms, color="tab:red", linewidth=1.5)

            title = "trial %d\nRT %.0f ms, %.1f deg" % (i, onset_ms, saccades["amplitude"][i])
        else:
            title = "trial %d\nNOT DETECTED" % i

        # The speed threshold used on this trial.
        onset = saccades["onset_index"][i]
        if not np.isnan(onset):
            baseline_lo, baseline_hi = settings["baseline_window"]
            in_baseline = (speed_time >= baseline_lo) & (speed_time <= baseline_hi)
            threshold = np.nanmean(speed[i][in_baseline]) + settings["velocity_threshold"]
            ax_speed.axhline(threshold, color="grey", linestyle="--", linewidth=1)

        for ax in (ax_pos, ax_speed):
            ax.axvline(0, color="black", linestyle=":", linewidth=1)

        ax_pos.set_title(title, fontsize=10)
        ax_speed.set_xlabel("Time from GO cue (ms)")

    axes[0, 0].set_ylabel("Eye position (deg)")
    axes[1, 0].set_ylabel("Eye speed (deg/s)")
    axes[0, 0].legend(fontsize=8, loc="upper left")

    fig.suptitle("Detection check: green = saccade onset, red = offset, "
                 "dotted = GO cue, dashed grey = speed threshold", y=1.01)
    fig.tight_layout()
    plt.show()


# Show the first six trials where something was detected.
detected_trials = np.where(saccades["detected"])[0]
plot_detection_check(eye_time, speed_time, smooth_x, smooth_y, eye_speed,
                     saccades, SETTINGS, detected_trials[:6])

Now the same thing for many trials at once. Each line is one trial's eye speed,
shifted so that **time 0 is that trial's own detected saccade onset** rather than
the GO cue. If the detector is working, all the speed peaks should stack up
neatly just after 0 instead of being smeared out.

In [ ]:
def compute_speed_aligned_to_saccade(speed, speed_time, saccades, ms_before=100, ms_after=150):
    """Re-cut the speed traces so that 0 is each trial's own saccade onset."""
    new_time = np.arange(-ms_before, ms_after + 1)
    n_trials = speed.shape[0]
    out = np.full((n_trials, len(new_time)), np.nan)

    for i in range(n_trials):
        if not saccades["detected"][i]:
            continue
        wanted = saccades["reaction_time"][i] + new_time
        out[i] = np.interp(wanted, speed_time, speed[i], left=np.nan, right=np.nan)

    return out, new_time


def plot_speed_overlay(speed_aligned, time_axis, n_show=100):
    """Overlay the saccade-aligned speed of many trials, plus their average."""
    fig, ax = plt.subplots(figsize=(8, 4.5))

    shown = 0
    for i in range(speed_aligned.shape[0]):
        if np.all(np.isnan(speed_aligned[i])):
            continue
        ax.plot(time_axis, speed_aligned[i], color="grey", linewidth=0.4, alpha=0.35)
        shown = shown + 1
        if shown >= n_show:
            break

    ax.plot(time_axis, np.nanmean(speed_aligned, axis=0), color="tab:red",
            linewidth=2.5, label="average of all detected trials")
    ax.axvline(0, color="tab:green", linewidth=1.5, label="detected saccade onset")

    ax.set_xlabel("Time from saccade onset (ms)")
    ax.set_ylabel("Eye speed (deg/s)")
    ax.set_title("%d individual trials, aligned on their own detected onset" % shown)
    ax.legend()
    fig.tight_layout()
    plt.show()


speed_aligned, saccade_time = compute_speed_aligned_to_saccade(eye_speed, speed_time, saccades)
plot_speed_overlay(speed_aligned, saccade_time)

---
# 8. The saccade parameters: what each number means

The detector returned one number per trial for each of these. Undetected trials are
`nan` throughout, so the arrays still line up with everything else.

| Parameter | Units | What it means |
|---|---|---|
| `reaction_time` | ms | **When the saccade started**, measured from the GO cue. The headline behavioural measure. Small = quick. |
| `saccade_end` | ms | When the eye landed, from the GO cue. |
| `duration` | ms | `saccade_end − reaction_time`. How long the movement took. Typically 27–55 ms here. |
| `amplitude` | degrees | **How far the eye travelled**, start point to end point. Should match the target eccentricity (15 or 20°). |
| `angle` | degrees | **Which way it travelled.** Same convention as `target_angle`: 0 = up, counter-clockwise, so 90 = left, 270 = right. |
| `peak_velocity` | deg/s | **The fastest the eye moved** during the saccade. Typically 1000–1500 deg/s. |
| `start_x`, `start_y` | degrees | Where the eye was when the saccade began — near the centre. |
| `end_x`, `end_y` | degrees | **Where the eye landed.** Compare with `target_x`, `target_y` for accuracy. |
| `detected` | True/False | Whether a saccade was found at all. Always check this first. |

## Three things about these numbers that surprise people

**1. Negative reaction times are real.** The delay between the target and the GO cue
was almost always 467 ms, so the monkey learned it and often started moving before
the GO cue formally arrived. A negative RT means "he jumped the gun". These are
genuine behaviour and should not be deleted — though remember that the trials where
he jumped *hardest* became fixation breaks and were dropped before the file was
written, so what you see is the mild end of the anticipation.

**2. RT and peak velocity can move independently.** They measure different things —
*when* the movement started versus *how fast* it went — and inactivating the FEF can
affect one without the other. Do not assume that "slower" means the same thing for both.

**3. Amplitude is the best single check on the detector.** The monkey was looking at
a target at a known distance. If the median amplitude comes out near the target
eccentricity, the detector is finding the real, target-directed saccade rather than
a blink or a small correction.

Let us look at the distributions.

In [ ]:
def plot_parameter_distributions(saccades, target_ecc):
    """Histograms of the four main saccade measures."""
    ok = saccades["detected"]

    panels = [
        ("reaction_time", "Reaction time (ms)", None),
        ("amplitude", "Amplitude (deg)", np.nanmedian(target_ecc)),
        ("peak_velocity", "Peak velocity (deg/s)", None),
        ("duration", "Duration (ms)", None),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))

    for ax, (key, label, reference) in zip(axes, panels):
        values = saccades[key][ok]
        ax.hist(values, bins=50, color="tab:blue", edgecolor="none")

        median = np.nanmedian(values)
        ax.axvline(median, color="black", linewidth=1.5)
        ax.set_title("%s\nmedian %.0f" % (label, median), fontsize=10)
        ax.set_xlabel(label)

        # For amplitude, also draw where the target actually was.
        if reference is not None:
            ax.axvline(reference, color="tab:red", linestyle="--", linewidth=1.5)
            ax.set_title("%s\nmedian %.1f (target at %.0f)" % (label, median, reference), fontsize=10)

    axes[0].set_ylabel("Number of trials")
    # Mark where "anticipated the GO cue" begins.
    axes[0].axvline(0, color="tab:red", linestyle="--", linewidth=1.5)
    fig.suptitle("Distributions of the saccade measures (red dashed = reference)", y=1.06)
    fig.tight_layout()
    plt.show()


plot_parameter_distributions(saccades, target_ecc)

n_anticipated = np.sum(saccades["reaction_time"][saccades["detected"]] < 0)
print("Saccades starting BEFORE the GO cue: %d of %d detected (%.0f%%)"
      % (n_anticipated, np.sum(saccades["detected"]),
         100 * n_anticipated / np.sum(saccades["detected"])))

## Check 1: did the saccades actually go towards the target?

This is the strongest single test that the detector found the *right* movement. We
compare the direction the eye travelled (`angle`) against where the target actually
was (`target_angle`).

The `np.mod(difference + 180, 360) - 180` below wraps the difference into the range
−180 to +180. Without it, a saccade at 5° and a target at 355° would look 350°
apart when they are really only 10° apart.

In [ ]:
def compute_angle_error(saccade_angle, target_angle):
    """How far off the target direction was each saccade, in degrees (-180..180)."""
    difference = saccade_angle - target_angle
    return np.mod(difference + 180, 360) - 180


angle_error = compute_angle_error(saccades["angle"], target_angle)

ok = saccades["detected"]
n_close = np.sum(np.abs(angle_error[ok]) < 30)

print("Median direction error: %.1f deg" % np.nanmedian(np.abs(angle_error[ok])))
print("Saccades within 30 deg of the target: %d of %d (%.1f%%)"
      % (n_close, np.sum(ok), 100 * n_close / np.sum(ok)))
print()
print("-> if that percentage is near 100, the detector is finding the real,")
print("   target-directed saccade and not a blink or a small correction.")

## Check 2: where did the eye actually land?

Each dot below is one trial's landing point. The crosses are the two target
positions. The dots should cluster tightly on the crosses.

In [ ]:
def plot_landing_points(saccades, target_x, target_y, is_left, is_right):
    """Scatter of where each saccade ended, with the true target positions marked."""
    fig, ax = plt.subplots(figsize=(6, 6))

    ok = saccades["detected"]

    groups = [("Leftward target", is_left & ok, "tab:blue"),
              ("Rightward target", is_right & ok, "tab:red")]

    for label, mask, colour in groups:
        ax.plot(saccades["end_x"][mask], saccades["end_y"][mask], ".",
                color=colour, markersize=3, alpha=0.3, label=label)

        # The true target position for this group.
        ax.plot(np.median(target_x[mask]), np.median(target_y[mask]), "X",
                color="black", markersize=14, markeredgecolor="white", markeredgewidth=1.5)

    ax.plot(0, 0, "+", color="black", markersize=14)   # the fixation point
    ax.set_xlabel("Horizontal eye position (deg)")
    ax.set_ylabel("Vertical eye position (deg)")
    ax.set_title("Where each saccade landed\n(X = true target, + = fixation point)")
    ax.axis("equal")
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_landing_points(saccades, target_x, target_y, is_left_saccade, is_right_saccade)

## Check 3: the main sequence

The **main sequence** is one of the most reliable facts about saccades: bigger
saccades are faster, following a tight, slightly curved relationship. Every healthy
oculomotor system shows it.

If your detected saccades fall on a clean curve here, they are real saccades. If
the plot is a shapeless cloud, the detector is picking up noise.

In [ ]:
def plot_main_sequence(saccades):
    """Peak velocity against amplitude -- the classic saccade main sequence."""
    ok = saccades["detected"]

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(saccades["amplitude"][ok], saccades["peak_velocity"][ok], ".",
            color="tab:purple", markersize=3, alpha=0.3)

    ax.set_xlabel("Amplitude (deg)")
    ax.set_ylabel("Peak velocity (deg/s)")
    ax.set_title("The main sequence: bigger saccades are faster")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()


plot_main_sequence(saccades)

## A toolkit for summarising by condition

Everything above described single saccades. To answer a scientific question you
need to summarise groups of trials and compare them, so the rest of this section
builds a small toolkit for exactly that. **You will use these four functions
constantly in the project (section 12)**, so it is worth reading them once.

We demonstrate them on **amplitude** — how far the saccade travelled. Amplitude is
a good one to start with because it is a *control* measure: the target never moved,
so amplitude should stay flat whatever the drug does. Seeing what "no effect" looks
like is the only way to recognise a real one later.

In [ ]:
def compute_bootstrap_ci(values, n_resamples=1000, low_pct=2.5, high_pct=97.5):
    """How uncertain is a median? Returns the lower and upper edge of its range.

    The idea, called bootstrapping, is simple: pretend our trials are the whole
    world, draw a fresh sample of the same size FROM them (allowing repeats), and
    take the median of that. Do it a thousand times and you get a thousand
    plausible medians. The middle 95% of those is the range the true median
    plausibly lies in.
    """
    values = values[~np.isnan(values)]
    if len(values) < 3:
        return np.nan, np.nan

    medians = np.full(n_resamples, np.nan)
    for k in range(n_resamples):
        picked = np.random.choice(values, size=len(values), replace=True)
        medians[k] = np.median(picked)

    return np.percentile(medians, low_pct), np.percentile(medians, high_pct)


def compute_summary_by_phase(values, valid, phases, groups):
    """Median of a per-trial measure, split by phase and by group.

    phases and groups are both lists of (name, mask) pairs, so this one function
    can split the data any way you like -- by injection phase and direction, by
    phase and object value, by early/late trials, and so on.

    Returns a plain list of rows. Computation only: no printing, no plotting.
    """
    rows = []

    for phase_name, phase_mask in phases:
        for group_name, group_mask in groups:
            selected = phase_mask & group_mask & valid
            chosen = values[selected]

            if np.any(selected):
                middle = np.nanmedian(chosen)
                low, high = compute_bootstrap_ci(chosen)
            else:
                middle, low, high = np.nan, np.nan, np.nan

            rows.append({"phase": phase_name, "group": group_name,
                         "n": int(np.sum(selected)),
                         "value": middle, "low": low, "high": high})

    return rows


def compute_proportion_by_phase(flag, valid, phases, groups):
    """Percentage of trials where flag is True, split by phase and group.

    Returns rows in the SAME shape as compute_summary_by_phase, so the same
    plotting function draws either one.
    """
    rows = []

    for phase_name, phase_mask in phases:
        for group_name, group_mask in groups:
            selected = phase_mask & group_mask & valid
            n = int(np.sum(selected))

            if n > 0:
                n_true = int(np.sum(flag & selected))
                percent = 100.0 * n_true / n
                # The uncertainty of a percentage, from the same bootstrap idea.
                spread = 100.0 * np.sqrt((percent / 100) * (1 - percent / 100) / n)
                low, high = percent - 1.96 * spread, percent + 1.96 * spread
            else:
                percent, low, high = np.nan, np.nan, np.nan

            rows.append({"phase": phase_name, "group": group_name, "n": n,
                         "value": percent, "low": low, "high": high})

    return rows


# The two ways we most often split the trials. Each is a list of (name, mask).
DIRECTION_GROUPS = [("leftward", is_left_saccade), ("rightward", is_right_saccade)]
VALUE_GROUPS = [("good object", is_good_object), ("bad object", is_bad_object)]

ALL_TRIALS = np.ones(n_trials, dtype=bool)   # useful when nothing should be excluded

Now the plotting workhorse. It takes whatever `compute_summary_by_phase` or
`compute_proportion_by_phase` returned and draws it as a grouped bar chart with
error bars. **Almost every question in section 10 is answered with this one
function**, so it is worth a look.

The error bars matter. A difference between two bars only means something if the
bars' error ranges do not overlap much — otherwise you are reading noise.

In [ ]:
def plot_summary_by_phase(rows, y_label, title, colours=None):
    """Grouped bar chart of the rows produced by the two compute_ functions above.

    Draws only what it is handed -- it computes nothing itself.
    """
    # Work out the phases and the groups, keeping the order they appear in.
    phase_names = []
    group_names = []
    for row in rows:
        if row["phase"] not in phase_names:
            phase_names.append(row["phase"])
        if row["group"] not in group_names:
            group_names.append(row["group"])

    if colours is None:
        colours = ["tab:blue", "tab:red", "tab:green", "tab:orange"]

    fig, ax = plt.subplots(figsize=(1.9 * len(phase_names) + 4, 4.5))

    bar_width = 0.8 / len(group_names)
    positions = np.arange(len(phase_names))

    for g, group_name in enumerate(group_names):
        heights, down, up, labels = [], [], [], []

        for phase_name in phase_names:
            # Find the one row matching this phase and this group.
            for row in rows:
                if row["phase"] == phase_name and row["group"] == group_name:
                    heights.append(row["value"])
                    # Error bars are given as distances from the top of the bar.
                    down.append(0 if np.isnan(row["low"]) else row["value"] - row["low"])
                    up.append(0 if np.isnan(row["high"]) else row["high"] - row["value"])
                    labels.append(row["n"])

        offset = (g - (len(group_names) - 1) / 2) * bar_width
        bars = ax.bar(positions + offset, heights, bar_width,
                      yerr=[down, up], capsize=4,
                      color=colours[g % len(colours)], label=group_name)

        # Write the value ABOVE the error bar (not on the bar, where the whisker
        # would strike through it), and the trial count inside the bar.
        for bar, height, up_error, n in zip(bars, heights, up, labels):
            if np.isnan(height):
                continue
            ax.text(bar.get_x() + bar.get_width() / 2, height + up_error,
                    "%.0f" % height, ha="center", va="bottom", fontsize=9)
            ax.text(bar.get_x() + bar.get_width() / 2, 0, "n=%d\n" % n,
                    ha="center", va="bottom", fontsize=7, color="white")

    # Leave headroom at the top so the legend has somewhere to sit that is not
    # on top of the bars.
    all_tops = [row["value"] + (0 if np.isnan(row["high"]) else row["high"] - row["value"])
                for row in rows if not np.isnan(row["value"])]
    if all_tops:
        highest = max(all_tops)
        lowest = min(0, min(row["value"] for row in rows if not np.isnan(row["value"])))
        ax.set_ylim(lowest * 1.1 if lowest < 0 else 0, highest * 1.3)

    ax.set_xticks(positions)
    ax.set_xticklabels(phase_names)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()


def plot_distributions_by_group(values, valid, groups, x_label, title, bins=45):
    """Overlaid histograms, one per group, drawn as outlines so they can overlap.

    A bar chart shows you two medians; this shows you the two whole
    distributions, which is how you tell a real shift from two similar spreads
    that happen to have slightly different middles.
    """
    fig, ax = plt.subplots(figsize=(8, 4.5))

    colours = ["tab:blue", "tab:red", "tab:green", "tab:orange"]
    for g, (group_name, group_mask) in enumerate(groups):
        chosen = values[group_mask & valid]
        chosen = chosen[~np.isnan(chosen)]
        if len(chosen) == 0:
            continue
        ax.hist(chosen, bins=bins, histtype="step", linewidth=2,
                color=colours[g % len(colours)],
                label="%s (median %.0f)" % (group_name, np.median(chosen)))
        ax.axvline(np.median(chosen), color=colours[g % len(colours)],
                   linestyle="--", linewidth=1)

    ax.set_xlabel(x_label)
    ax.set_ylabel("Number of trials")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()


# Amplitude, split by injection phase and saccade direction.
amplitude_rows = compute_summary_by_phase(saccades["amplitude"], saccades["detected"],
                                          injection_phases, DIRECTION_GROUPS)

plot_summary_by_phase(amplitude_rows, "Median amplitude (deg)",
                      "Amplitude by injection phase and direction (a control measure)")

The bars barely move, and every error bar overlaps every other one. That is what a
measure with **no effect** looks like, and it is the right sanity check: the target
never moved, so if amplitude had changed we would suspect the analysis before
believing the biology.

Now swap `saccades["amplitude"]` for another measure and the same two lines answer
a different question. That is what section 12 asks you to do.

If you want the numbers as a table too, `pandas` will lay the same rows out. We use
it only for printing, never to do the analysis.

In [ ]:
table = pd.DataFrame(amplitude_rows).pivot(index="group", columns="phase", values="value")
table = table[[name for name, mask in injection_phases]]     # keep them in time order

print("MEDIAN AMPLITUDE (deg)")
print(table.round(2))

## Measures across the session, trial by trial

Bars collapse a whole phase into one number, which hides *when* things changed. The
drug washes in and out gradually, so the shape over trials matters. This pair of
functions draws any per-trial measure against trial number, with the injection
phases shaded behind it.

In [ ]:
def compute_running_average(trial_numbers, values, mask, window, method="median"):
    """A running summary of a measure over trials, within one group of trials.

    method="median" is right for most measures -- it ignores the odd wild trial.
    method="mean" is right for FIRING RATES, and the reason is worth knowing: a
    rate measured in a 200 ms window can only come out as 0, 5, 10, 15 ... 
    spikes/s, because you cannot have half a spike. A running median of numbers
    that only take a few values just snaps to those values, so the line comes out
    as a staircase. A running mean lands between them and reads as a smooth curve.

    Returns the trial numbers and the smoothed values, for plotting.
    """
    group_trials = trial_numbers[mask]
    group_values = values[mask]

    smoothed = np.full(len(group_values), np.nan)

    for k in range(len(group_values)):
        lo = max(0, k - window // 2)
        hi = min(len(group_values), k + window // 2 + 1)
        chunk = group_values[lo:hi]
        if not np.all(np.isnan(chunk)):
            if method == "mean":
                smoothed[k] = np.nanmean(chunk)
            else:
                smoothed[k] = np.nanmedian(chunk)

    return group_trials, smoothed


def plot_session_course(trial_numbers, values, groups, phases, y_label, title,
                        window=25, method="median"):
    """Draw a measure across the whole session, one line per group, phases shaded.

    The y axis is scaled to the SMOOTHED lines plus the bulk of the raw trials,
    not to the most extreme trial. A handful of outliers would otherwise stretch
    the axis until the line you care about was a flat squiggle at the bottom.
    A few extreme points may therefore sit outside the visible range.

    Use method="mean" for firing rates -- see compute_running_average.
    """
    fig, ax = plt.subplots(figsize=(12, 4.5))

    # Draw the data first, and remember the range that actually matters.
    lowest, highest = [], []

    for label, mask, colour in groups:
        # The raw trials, faint.
        ax.plot(trial_numbers[mask], values[mask], ".", color=colour,
                markersize=2, alpha=0.18, zorder=1)
        # The running summary, bold.
        x, y = compute_running_average(trial_numbers, values, mask, window, method)
        ax.plot(x, y, color=colour, linewidth=2, label=label, zorder=2)

        # The smoothed line must always be fully visible...
        if not np.all(np.isnan(y)):
            lowest.append(np.nanmin(y))
            highest.append(np.nanmax(y))
        # ...and so should the bulk of the raw points, but not their extremes.
        raw = values[mask]
        raw = raw[~np.isnan(raw)]
        if len(raw) > 0:
            lowest.append(np.percentile(raw, 1))
            highest.append(np.percentile(raw, 99))

    if lowest:
        low, high = min(lowest), max(highest)
        padding = 0.08 * (high - low) if high > low else 1.0
        ax.set_ylim(low - padding, high + padding)

    # Make a little empty space at the top so the phase names have somewhere to
    # sit without landing on the data or the title.
    y_low, y_high = ax.get_ylim()
    height = y_high - y_low
    ax.set_ylim(y_low, y_high + 0.12 * height)
    label_y = y_high + 0.09 * height

    # Now shade the injection phases behind everything, and name each one.
    shades = {"before": "white", "during": "#ffe8b0", "after": "#cfe6ff",
              "after 2nd injection": "#d8f0d0"}
    for phase_name, phase_mask in phases:
        phase_trials = trial_numbers[phase_mask]
        ax.axvspan(phase_trials.min(), phase_trials.max(),
                   color=shades.get(phase_name, "#eeeeee"), alpha=0.7, zorder=0)
        ax.text(np.median(phase_trials), label_y, phase_name,
                ha="center", va="center", fontsize=10)

    ax.set_xlabel("Trial number (in the order they were recorded)")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend(loc="best")
    fig.tight_layout()
    plt.show()


trial_numbers = np.arange(n_trials)
ok = saccades["detected"]

direction_groups = [("Leftward saccade", is_left_saccade & ok, "tab:blue"),
                    ("Rightward saccade", is_right_saccade & ok, "tab:red")]

plot_session_course(trial_numbers, saccades["amplitude"], direction_groups,
                    injection_phases, "Amplitude (deg)",
                    "Amplitude across the session (running median of 25 trials)")

---
# 9. More behavioural measures

RT and peak velocity are not the only things a saccade can tell you. This section
adds four more read-outs. **The functions are written for you** — in the project
you only have to choose what to plot with them.

Each one was chosen because it separates something the others confound.

## Accuracy: which way did the eye miss?

A plain distance from the landing point to the target throws away the thing you
most want to know — **which way** it missed — and worse, it cannot cancel. A group
scattering evenly around the target reads the same as one landing consistently
short, because two opposite misses average to a large error rather than to no bias.

So we split the miss into two signed numbers, in a frame built per trial from the
fixation point towards that trial's target:

- **radial error** — positive = past the target (overshoot), negative = short of it
  (**undershoot**, or *hypometria*, the classic sign of a weakened saccade)
- **tangential error** — sideways off the line to the target

Building the frame per trial is what makes left and right comparable: "short" then
means the same thing on both sides, even though the targets point opposite ways.

In [ ]:
def compute_endpoint_errors(end_x, end_y, target_x, target_y):
    """How far, and in which direction, each saccade missed its target.

    Returns radial error (+ = past the target, - = short of it) and tangential
    error (sideways), both in degrees, measured along each trial's own
    fixation-to-target axis.
    """
    # A unit vector pointing from the fixation point towards the target.
    target_distance = np.hypot(target_x, target_y)
    unit_x = target_x / target_distance
    unit_y = target_y / target_distance

    # Where the eye landed, relative to the target.
    miss_x = end_x - target_x
    miss_y = end_y - target_y

    # Split that miss into "along the axis" and "across the axis".
    radial = miss_x * unit_x + miss_y * unit_y
    tangential = -miss_x * unit_y + miss_y * unit_x

    return radial, tangential


radial_error, tangential_error = compute_endpoint_errors(
    saccades["end_x"], saccades["end_y"], target_x, target_y)

ok = saccades["detected"]
print("Radial error   : median %+.2f deg  (negative = fell short of the target)"
      % np.nanmedian(radial_error[ok]))
print("Tangential error: median %+.2f deg" % np.nanmedian(tangential_error[ok]))

## Precision: how repeatable was the landing?

Accuracy and precision are different. A set of saccades can be **biased but tight**
(always 2° short, but always in the same place) or **centred but sloppy** (right on
average, scattered everywhere). Only having both tells you which, and inactivation
can produce either.

Scatter is a property of a *group*, not of a trial — there is no per-trial value to
average — so it returns one number per group and carries no error bar.

In [ ]:
def compute_endpoint_scatter(end_x, end_y, selected):
    """How spread out the landing points were, in degrees.

    This is the 2-D standard distance: the spread about the group's own centre,
    combining horizontal and vertical. One number per GROUP of trials.
    """
    x = end_x[selected]
    y = end_y[selected]
    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]
    if len(x) < 3:
        return np.nan
    return np.sqrt(np.var(x) + np.var(y))


for phase_name, phase_mask in injection_phases:
    scatter = compute_endpoint_scatter(saccades["end_x"], saccades["end_y"],
                                       phase_mask & is_right_saccade & ok)
    print("%-8s rightward endpoint scatter: %.2f deg" % (phase_name, scatter))

## Seeing the endpoints in two dimensions

Everything so far collapsed each saccade to a single number. The landing points do
not have to be: they live on a screen, and a **2-D density map** shows accuracy,
bias and scatter all at once, in a way three separate numbers never quite do.

This is the first two-dimensional plot in the notebook, so here is how it works.

**Binning in 2-D.** A histogram counts how many values fall in each bin along one
axis. A 2-D histogram lays a grid over the screen and counts how many landing
points fall in each square. `np.histogram2d` does it, and the result is just a
matrix of counts that we draw as an image with `imshow`.

**Three details that decide whether the picture is honest:**

1. **Percentages, not counts.** The phases have very different numbers of trials
   (300 before, 1120 after here). Raw counts would make "before" look almost empty
   next to "after" even if the behaviour were identical. Dividing by each group's
   own trial count fixes that, so every panel says "% of *that group's* trials".
2. **One colour scale for all the panels.** If each panel scaled to its own
   maximum, a faint blob and a dense one would look the same. Sharing the scale is
   what makes the panels comparable at a glance.
3. **Square panels.** One degree horizontally must be the same length as one degree
   vertically, or the scatter looks stretched in whichever direction the figure is
   wider. `ax.set_aspect("equal")` enforces it.

We smooth the grid slightly, because with 1° bins a raw count map is speckled and
the shape is hard to see.

In [ ]:
from scipy.signal import convolve2d


def make_gaussian_kernel(smooth_bins):
    """A small 2-D gaussian blur, built by hand. Returns None for no smoothing."""
    if smooth_bins <= 0:
        return None

    half = int(np.ceil(2 * smooth_bins))
    offsets = np.arange(-half, half + 1)
    grid_x, grid_y = np.meshgrid(offsets, offsets)

    kernel = np.exp(-(grid_x ** 2 + grid_y ** 2) / (2 * smooth_bins ** 2))
    return kernel / np.sum(kernel)


def compute_endpoint_heatmap(end_x, end_y, groups, edges, smooth_bins=1.2):
    """Where the eye landed, as a 2-D density for each group.

    Returns a list of dictionaries with the group's name, its density grid and
    its trial count. The density is the PERCENTAGE of that group's own trials per
    bin, so groups of different size stay comparable.

    Computation only -- it draws nothing.
    """
    kernel = make_gaussian_kernel(smooth_bins)
    maps = []

    for group_name, group_mask in groups:
        selected = group_mask & ~np.isnan(end_x) & ~np.isnan(end_y)
        n = int(np.sum(selected))

        counts, _, _ = np.histogram2d(end_x[selected], end_y[selected],
                                      bins=[edges, edges])
        density = 100.0 * counts / max(n, 1)

        if kernel is not None:
            density = convolve2d(density, kernel, mode="same")

        maps.append({"name": group_name, "density": density, "n": n})

    return maps


def plot_endpoint_heatmaps(maps, edges, target_x, target_y, title, n_columns=None):
    """Draw the endpoint densities as a row of heatmaps sharing one colour scale."""
    if n_columns is None:
        n_columns = len(maps)
    n_rows = int(np.ceil(len(maps) / n_columns))

    fig, axes = plt.subplots(n_rows, n_columns,
                             figsize=(3.6 * n_columns, 3.9 * n_rows),
                             squeeze=False)

    # One scale for every panel, so a colour means the same thing everywhere.
    peak = max(np.max(m["density"]) for m in maps)
    if peak <= 0:
        peak = 1.0

    # Where the two targets were, to mark on every panel.
    target_spots = np.unique(np.round(np.column_stack([target_x, target_y]), 1), axis=0)

    for k, m in enumerate(maps):
        ax = axes[k // n_columns][k % n_columns]

        # .T because histogram2d puts x along the first axis, while an image
        # wants rows to be y. origin="lower" keeps up on the screen as up here.
        picture = ax.imshow(m["density"].T, origin="lower", cmap="hot",
                            vmin=0, vmax=peak,
                            extent=[edges[0], edges[-1], edges[0], edges[-1]])

        ax.plot(0, 0, "+", color="white", markersize=9)          # fixation point
        ax.plot(target_spots[:, 0], target_spots[:, 1], "o",
                markerfacecolor="none", markeredgecolor="white", markersize=11)

        ax.set_aspect("equal")       # a degree must look the same in x and y
        ax.set_title("%s (n=%d)" % (m["name"], m["n"]), fontsize=10)
        ax.set_xlabel("Eye X (deg)")
        if k % n_columns == 0:
            ax.set_ylabel("Eye Y (deg)")

    # Hide any unused panels in the grid.
    for k in range(len(maps), n_rows * n_columns):
        axes[k // n_columns][k % n_columns].axis("off")

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    bar = fig.colorbar(picture, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    bar.set_label("% of that group's trials per bin")
    plt.show()


# The grid, sized from the target eccentricity so it fits any session. 1.4x
# leaves room for an overshoot without wasting most of the panel on empty screen.
grid_limit = int(np.ceil(np.nanmedian(target_ecc) * 1.4))
endpoint_edges = np.arange(-grid_limit, grid_limit + 1, 1.0)

maps = compute_endpoint_heatmap(saccades["end_x"], saccades["end_y"],
                                injection_phases, endpoint_edges)

plot_endpoint_heatmaps(maps, endpoint_edges, target_x, target_y,
                       "Where the eye landed, by injection phase "
                       "(+ = fixation point, o = targets)")

Read that figure the way you would read any map. The two bright blobs are the two
target locations. Ask three questions of it:

- **Are the blobs centred on the circles?** If a blob sits between the circle and
  the fixation cross, those saccades fell short — hypometria.
- **Is one blob more spread out than the other?** That is precision, and it can
  change on one side only.
- **Does either change across the panels?** That is the inactivation effect, seen
  directly rather than through a summary number.

Both directions share a panel on purpose: they land on opposite sides and cannot
overlap, so one panel per phase shows everything, and the two sides sit on the same
colour scale where they can be compared honestly.

## How long did the eye stay on the object?

After landing, the monkey holds gaze on the object until allowed to look away. The
task enforces only a **minimum** hold, so how long he *actually* holds is free
behaviour — and a very plausible place for object value to show up, because a
valuable object is worth looking at.

### This measure needs its own, much longer alignment

The hold is around 800 ms and starts about 150 ms after the GO cue, while the
window we used for RT ends at +600 ms. Measured in that window, **96% of trials get
cut off** and you end up measuring the window rather than the monkey. So we align
again with a long post-window.

Two things can end a hold artificially, and both must be reported, not buried:

- **Censored** — the data ran out while the eye was still on target. Those are
  *lower bounds*, so they are excluded from comparisons.
- **At ceiling** — the hold ended because the target disappeared. The monkey might
  have held longer given the chance. This hits the *longest-holding* condition
  hardest, so it can shrink a real difference rather than create one.

### Why the window is 7 degrees

The task's own acceptance window is not stored in the file, so it has to be
inferred — and it must be, because **the window is the measurement here**. An
earlier 5° setting captured only about three quarters of compliant trials, and the
misses were not random: saccades after the injection land further off centre, so a
tight window scores an inaccurate-but-compliant hold as broken and manufactures a
"shorter hold" exactly where the drug makes saccades least accurate. At 7° almost
every rewarded trial is counted.

In [ ]:
GAZE_POST_MS = 1800     # long enough that almost no hold gets cut off
GAZE_WINDOW_DEG = 7     # distance from the target counted as "still on it"

# A second alignment, with a much longer post-window. Note we use the RAW aligned
# positions here, not the smoothed ones: this measure only asks "is the eye near
# the target?", which needs no smoothing, and smoothing would throw away trials
# that happen to contain a blink later in the trial.
gaze_x, gaze_y, gaze_time = align_eye_traces(
    eye_x, eye_y, eye_n_samples, eye_bin_width, go_cue_time, MS_BEFORE, GAZE_POST_MS)


def compute_gaze_hold(gaze_x, gaze_y, gaze_time, target_x, target_y, start_ms,
                      window_deg=7, max_gap_ms=20, search_ms=50):
    """How long the eye stayed near the target after the saccade landed.

    Starts at the end of each saccade, calls the eye "on target" while it is
    within window_deg, and walks forward until it leaves and STAYS away for
    longer than max_gap_ms -- so a blink or a small correction does not end
    the hold.

    Returns a dictionary of per-trial arrays.
    """
    n_trials = gaze_x.shape[0]

    on_target = np.zeros(n_trials, dtype=bool)
    hold_start = np.full(n_trials, np.nan)
    hold_end = np.full(n_trials, np.nan)
    censored = np.zeros(n_trials, dtype=bool)

    gap_samples = max(1, int(round(max_gap_ms / np.median(np.diff(gaze_time)))))

    for i in range(n_trials):
        if np.isnan(start_ms[i]):
            continue

        # Distance from the eye to THIS trial's target, at every sample.
        distance = np.hypot(gaze_x[i] - target_x[i], gaze_y[i] - target_y[i])
        near = distance <= window_deg          # a nan distance compares False

        # When does the hold begin? The first on-target sample after the saccade
        # ended. The short search window allows for the detected offset sitting a
        # sample or two before the eye has fully settled.
        candidates = np.where(near & (gaze_time >= start_ms[i])
                                   & (gaze_time <= start_ms[i] + search_ms))[0]
        if len(candidates) == 0:
            continue                            # never landed on the target

        first = candidates[0]
        on_target[i] = True
        hold_start[i] = gaze_time[first]

        # When does it end? Walk forward, tolerating brief excursions.
        last_near = first
        k = first
        while k < len(near) - 1:
            k = k + 1
            if near[k]:
                last_near = k
            elif (k - last_near) > gap_samples:
                break

        hold_end[i] = gaze_time[last_near]

        # Did the hold end because the monkey looked away, or because the
        # recording stopped? Keyed on the last sample that HAS data, so it
        # catches both the window ending and the trial's own trace ending.
        has_data = np.where(~np.isnan(distance))[0]
        censored[i] = last_near >= has_data[-1] - gap_samples

    return {"on_target": on_target, "hold_start": hold_start, "hold_end": hold_end,
            "hold_duration": hold_end - hold_start, "censored": censored}


gaze = compute_gaze_hold(gaze_x, gaze_y, gaze_time, target_x, target_y,
                         saccades["saccade_end"], GAZE_WINDOW_DEG)

# When did the target itself disappear? A hold that ended then was stopped by the
# task, not by the monkey.
target_off_time = find_event_time(event_codes, event_times, 26)    # TGTOFF
target_off_relative = target_off_time - go_cue_time

CEILING_TOLERANCE = 50      # ms
at_ceiling = gaze["on_target"] & ~gaze["censored"] & \
             (np.abs(gaze["hold_end"] - target_off_relative) < CEILING_TOLERANCE)

# Only uncensored holds are comparable: a censored one is a lower bound, and if
# censoring differs between groups the comparison is biased, not merely noisy.
gaze_valid = gaze["on_target"] & ~gaze["censored"]

print("Landed on target : %d of %d detected saccades (%.1f%%)"
      % (np.sum(gaze["on_target"]), np.sum(ok), 100 * np.sum(gaze["on_target"]) / np.sum(ok)))
print("Hold duration    : median %.0f ms  [5th-95th pct %.0f to %.0f]"
      % (np.nanmedian(gaze["hold_duration"]),
         np.nanpercentile(gaze["hold_duration"], 5),
         np.nanpercentile(gaze["hold_duration"], 95)))
print("Censored         : %d (%.1f%%)  <- excluded from comparisons"
      % (np.sum(gaze["censored"]), 100 * np.sum(gaze["censored"]) / np.sum(gaze["on_target"])))
print("At ceiling       : %d (%.1f%%)  <- ended when the target vanished"
      % (np.sum(at_ceiling), 100 * np.sum(at_ceiling) / np.sum(gaze["on_target"] & ~gaze["censored"])))
print("Target disappeared %.0f ms after the GO cue (median)" % np.nanmedian(target_off_relative))

### One more gaze measure, immune to both problems

"Did the monkey still have the target when it disappeared?" is a yes/no that every
trial answers the same way, so it has no ceiling and no censoring problem. It is
the safest of the gaze measures, at the cost of being coarser.

In [ ]:
# A trial counts only if we can actually see up to target offset.
gaze_seen_to_offset = (gaze["on_target"]
                       & (gaze["hold_start"] < target_off_relative)
                       & (~gaze["censored"]
                          | (gaze["hold_end"] >= target_off_relative - CEILING_TOLERANCE)))

held_to_offset = gaze["on_target"] & (gaze["hold_end"] >= target_off_relative - CEILING_TOLERANCE)

print("Usable for this measure: %d of %d on-target trials"
      % (np.sum(gaze_seen_to_offset), np.sum(gaze["on_target"])))
print("Still on target at target offset: %.1f%% of them"
      % (100 * np.sum(held_to_offset & gaze_seen_to_offset) / np.sum(gaze_seen_to_offset)))

## Everything in one place

Finally we gather every per-trial behavioural measure into one dictionary, so the
project can reach for any of them the same way: `behaviour["radial_error"]`,
`behaviour["hold_duration"]`, and so on.

Each measure comes with a `valid` mask saying which trials it can honestly be
measured on — they are **not** all the same, which is exactly why they are kept
together with the measures rather than assumed.

In [ ]:
ANTICIPATION_MS = 0     # a saccade starting before this counts as anticipatory

behaviour = {
    "reaction_time": saccades["reaction_time"],
    "peak_velocity": saccades["peak_velocity"],
    "duration": saccades["duration"],
    "amplitude": saccades["amplitude"],
    "radial_error": radial_error,
    "tangential_error": tangential_error,
    "hold_duration": gaze["hold_duration"],
    "anticipated": saccades["reaction_time"] < ANTICIPATION_MS,
    "held_to_offset": held_to_offset,
}

# Which trials each measure may honestly be read on.
valid_for = {
    "reaction_time": saccades["detected"],
    "peak_velocity": saccades["detected"],
    "duration": saccades["detected"],
    "amplitude": saccades["detected"],
    "radial_error": saccades["detected"],
    "tangential_error": saccades["detected"],
    "hold_duration": gaze_valid,               # censored trials excluded
    "anticipated": saccades["detected"],
    "held_to_offset": gaze_seen_to_offset,     # only trials we can see that far
}

UNITS = {"reaction_time": "ms", "peak_velocity": "deg/s", "duration": "ms",
         "amplitude": "deg", "radial_error": "deg", "tangential_error": "deg",
         "hold_duration": "ms", "anticipated": "%", "held_to_offset": "%"}

print("%-18s %8s  %s" % ("measure", "usable", "units"))
for name in behaviour:
    print("%-18s %8d  %s" % (name, np.sum(valid_for[name]), UNITS[name]))

---
# 10. The neuron: rasters and PSTHs

One FEF neuron was recorded throughout. `spike_times[i]` holds the times of every
spike on trial `i`, in ms from that trial's start, padded with `nan`.

Two standard ways of looking at a neuron:

- A **raster** draws one row per trial and one tick per spike. You see the raw data.
- A **PSTH** (peri-stimulus time histogram) counts the spikes into small time bins
  and averages over trials, giving firing rate in spikes per second. You see the
  pattern.

Both need the spikes **aligned** to an event first — exactly the same subtraction
idea as for the eye traces.

## Why we align to two different events

We will align the same spikes twice: once to **target onset**, once to the **saccade
onset**. Comparing the two answers a real question about what this neuron does.

- If the neuron responds to *seeing* the target, its response is locked to target
  onset. Aligned to the target it looks sharp; aligned to the saccade it smears out,
  because the reaction time varies from trial to trial.
- If the neuron drives the *movement*, the opposite happens.

So putting the two side by side separates a **visual** response from a **motor** one.

In [ ]:
def align_spikes(spike_times, align_time, window, bin_width=1):
    """Re-express every spike as time from a marker, and count them into bins.

    Returns:
      bin_centres   the time axis, in ms from the marker
      counts        trials x bins, the number of spikes in each bin
      per_trial     a list, one entry per trial, of that trial's aligned spike times
                    (this is what a raster plot needs)
    """
    n_trials = spike_times.shape[0]

    edges = np.arange(window[0], window[1] + bin_width, bin_width)
    bin_centres = edges[:-1] + bin_width / 2.0

    counts = np.full((n_trials, len(bin_centres)), np.nan)
    per_trial = []

    for i in range(n_trials):
        if np.isnan(align_time[i]):
            per_trial.append(np.array([]))
            continue          # no marker: leave this row as nan, NOT as zero

        # This trial's spikes, padding removed, shifted so 0 is the marker.
        trial_spikes = spike_times[i]
        trial_spikes = trial_spikes[~np.isnan(trial_spikes)] - align_time[i]

        # Keep only the ones inside our window, for the raster.
        in_window = (trial_spikes >= window[0]) & (trial_spikes < window[1])
        per_trial.append(trial_spikes[in_window])

        counts[i] = np.histogram(trial_spikes, edges)[0]

    return bin_centres, counts, per_trial


def compute_psth(bin_centres, counts, mask, bin_width=1, smooth_ms=20):
    """Average firing rate over a group of trials, in spikes per second."""
    group_counts = counts[mask]

    # Mean spikes per bin, then convert to spikes per second.
    mean_per_bin = np.nanmean(group_counts, axis=0)
    rate = mean_per_bin * (1000.0 / bin_width)

    # Smooth it a little, otherwise 1 ms bins are far too spiky to read.
    kernel = np.ones(smooth_ms) / smooth_ms
    return np.convolve(rate, kernel, mode="same")

An important detail is hidden in `align_spikes`: a trial with no marker gets a row
of `nan`, **not a row of zeros**. "We could not measure this trial" and "this trial
had no spikes" are completely different statements, and writing zero would quietly
drag the average down.

Now the figure. Two columns — target-aligned on the left, saccade-aligned on the
right — with the raster above and the PSTH below.

In [ ]:
def plot_spike_overview(alignments, is_left, is_right, n_raster_trials=150):
    """Raster and PSTH, one column per alignment.

    alignments is a list of (name, bin_centres, counts, per_trial) tuples.
    """
    n_columns = len(alignments)
    fig, axes = plt.subplots(2, n_columns, figsize=(6.5 * n_columns, 7),
                             gridspec_kw={"height_ratios": [2, 1]})

    groups = [("Leftward", is_left, "tab:blue"), ("Rightward", is_right, "tab:red")]

    for column, (name, bin_centres, counts, per_trial) in enumerate(alignments):
        ax_raster = axes[0, column]
        ax_psth = axes[1, column]

        # --- the raster: one row of dots per trial, grouped by direction ---
        row = 0
        for label, mask, colour in groups:
            trials_here = np.where(mask)[0][:n_raster_trials]
            for i in trials_here:
                spikes = per_trial[i]
                if len(spikes) > 0:
                    ax_raster.plot(spikes, np.full(len(spikes), row), "|",
                                   color=colour, markersize=2.5, markeredgewidth=0.6)
                row = row + 1
            # A line separating the two direction groups.
            ax_raster.axhline(row, color="black", linewidth=0.8)

        ax_raster.set_ylim(row, -1)
        ax_raster.set_ylabel("Trial (grouped by direction)")
        ax_raster.set_title("Aligned to %s" % name)
        ax_raster.axvline(0, color="black", linestyle="--", linewidth=1.2)

        # --- the PSTH below ---
        for label, mask, colour in groups:
            rate = compute_psth(bin_centres, counts, mask)
            ax_psth.plot(bin_centres, rate, color=colour, linewidth=1.8, label=label)

        ax_psth.axvline(0, color="black", linestyle="--", linewidth=1.2)
        ax_psth.set_xlabel("Time from %s (ms)" % name)
        ax_psth.set_ylabel("Firing rate (spikes/s)")
        ax_psth.legend(fontsize=9)

    # Give both PSTHs the same y axis, so the columns can be compared fairly.
    psth_axes = [axes[1, c] for c in range(n_columns)]
    top = max(ax.get_ylim()[1] for ax in psth_axes)
    for ax in psth_axes:
        ax.set_ylim(0, top)

    fig.tight_layout()
    plt.show()


# The saccade onset in TRIAL time: the RT is measured from the GO cue, so adding
# the GO cue time puts it back on the trial's own clock.
saccade_onset_time = go_cue_time + saccades["reaction_time"]

target_centres, target_counts, target_spikes = align_spikes(
    spike_times, target_on_time, window=(-300, 600))

saccade_centres, saccade_counts, saccade_spikes = align_spikes(
    spike_times, saccade_onset_time, window=(-300, 600))

plot_spike_overview(
    [("target onset", target_centres, target_counts, target_spikes),
     ("saccade onset", saccade_centres, saccade_counts, saccade_spikes)],
    is_left_saccade, is_right_saccade)

## Measuring the firing rate in a window

To put numbers on it, count the spikes in a chosen window and divide by how long the
window was. Two windows are standard here:

- a **baseline** window before the event, to say what the neuron does at rest
- a **response** window just after it, to catch the reaction

⚠️ **A firing rate is meaningless unless you say what it was aligned to.** A
"baseline" of −200 to 0 ms before the *target* is genuine rest; the same window
before the *GO cue* falls in the middle of the delay period, when the neuron is
already responding to the target it can see. On this session those two give
5.2 and 15.7 spikes/s — a threefold difference from the same window name. Always
state the alignment.

In [ ]:
def compute_firing_rate(bin_centres, counts, window, bin_width=1):
    """Firing rate of every trial inside a time window, in spikes per second.

    Trials with no marker (an all-nan row) return nan, so they drop out of an
    average instead of counting as a zero-spike trial.
    """
    in_window = (bin_centres >= window[0]) & (bin_centres < window[1])

    n_spikes = np.nansum(counts[:, in_window], axis=1)

    # Only count bins that actually hold data.
    n_valid_bins = np.sum(~np.isnan(counts[:, in_window]), axis=1)
    seconds_observed = n_valid_bins * bin_width / 1000.0

    rate = np.full(counts.shape[0], np.nan)
    measured = n_valid_bins > 0
    rate[measured] = n_spikes[measured] / seconds_observed[measured]
    return rate


baseline_rate = compute_firing_rate(target_centres, target_counts, (-200, 0))
visual_rate = compute_firing_rate(target_centres, target_counts, (50, 250))

print("Firing rate, aligned to TARGET ONSET")
print("  baseline (-200 to 0 ms):  %.1f spikes/s" % np.nanmean(baseline_rate))
print("  response (50 to 250 ms):  %.1f spikes/s" % np.nanmean(visual_rate))
print()
print("How these change across the injection is the first question of the project.")

---
# 11. Statistics: telling a real difference from noise

Every figure so far shows a difference. None of them yet says whether that
difference is bigger than the noise. That is what this section adds, and it is
what turns a plot into a claim you can defend in a report.

Three functions, each answering a different question.

| Function | The question it answers |
|---|---|
| `compare_two_groups` | Is group A different from group B? |
| `compare_change` | Did this group change from one phase to another? |
| `compare_interaction` | Did the change **differ between two groups**? |

The first two are both Mann-Whitney tests — the same tool, pointed at different
pairs of trials. The third is **not**, and it is worth understanding why, because
the temptation to reuse Mann-Whitney there is strong and the mistake is common.

### Why the interaction needs a different test

A Mann-Whitney test compares **two** samples. An interaction is a question about
**four**: it asks whether the change from before to during is *bigger in one group
than in another*. That quantity — a difference of differences — is not a two-sample
rank comparison, so no single Mann-Whitney can produce it.

The tempting shortcut is to run Mann-Whitney twice and compare the verdicts:

> "Rightward changed, p < 0.001. Leftward did not change, p = 0.68.
> Therefore the drug affected rightward more than leftward."

**That reasoning is invalid**, and it is one of the most common errors in the
literature. Two p-values landing on opposite sides of 0.05 does not mean the two
effects differ. Imagine rightward changing by +35 ms with p = 0.001 and leftward by
+30 ms with p = 0.06 — nearly the same effect, opposite verdicts, decided by
nothing more than how many trials happened to be in each group. The comparison you
want has to be measured directly, with its own error bar.

### "Can a p-value correction fix that?"

No — and it is worth being clear why, because the two problems get confused.

A correction (Bonferroni, FDR) fixes a **different** problem: running many tests
inflates the chance that one fires by accident, so it makes each test more
conservative. It does nothing about the fallacy above. Correcting the two p-values
here turns 1.2e-18 and 0.68 into 2.4e-18 and 1.0 — still two separate numbers,
neither of which is an estimate of *how much more* one group changed than the other.

The problem is not that the p-values are too generous. It is that **the quantity
you care about was never computed.** No correction can produce an estimate of a
contrast you did not measure.

### "What about an ANOVA?"

**Yes.** A two-way ANOVA — phase × direction — has an interaction term that tests
exactly this, and it is the classical way to do it. On this session all three
routes agree:

| Method | Interaction |
|---|---|
| Two-way ANOVA on the raw RT | F = 31.7, p < 0.001 |
| ANOVA on the ranks | F = 36.5, p < 0.001 |
| Permutation test on medians | +31.5 ms, p = 0.0002 |

So the permutation test is a **choice, not a necessity**. Three reasons for it here,
none of them "ANOVA is wrong":

1. **It matches what the figures show.** Every bar in this notebook is a *median*.
   An ANOVA tests *means*. Testing one quantity while plotting another invites a
   report where the figure and the statistic disagree.
2. **It assumes nothing.** RT here fails a normality test (Shapiro-Wilk
   p = 4e-07) and the four groups fail an equal-variance test (Levene p = 0.025).
   With ~150 trials per cell an ANOVA is robust enough that it survives this — as
   the table shows — but you have to *check* that, whereas a permutation test never
   needed the assumption.
3. **It gives you the effect size in milliseconds.** `F = 31.7` does not tell you
   whether the extra slowing was 3 ms or 30 ms. `+31.5 ms` does.

That third point is the one that matters most for your report. **Report the size of
the effect, not only whether it passed a threshold.**

Both are provided below. If your advisor expects an ANOVA, run the ANOVA — and if
the two disagree, that disagreement is itself informative and worth investigating
rather than picking whichever you prefer.

So we use **Mann-Whitney for every two-group comparison** (Step 1 of the project
leans on it heavily), and either a **permutation test** or an **ANOVA** for the
interaction.

## Why a median and a rank test

Reaction times are not bell-shaped: this task mixes anticipatory and reactive
trials, so the distribution has a long tail and a lump near zero. A mean gets
dragged by the tail, and tests that assume a bell shape are not safe. So we use the
**median**, and the **Mann-Whitney U test**, which only compares ranks and makes no
assumption about the shape.

In [ ]:
from scipy.stats import mannwhitneyu


def compute_difference_ci(values, mask_a, mask_b, n_resamples=1000):
    """The difference between two groups' medians, with a bootstrap 95% range.

    Resample each group independently, take the difference of the two medians,
    and repeat. If the resulting range does not cross zero, the difference is
    bigger than the noise. Returns (difference, low, high), where the difference
    is group B minus group A.
    """
    a = values[mask_a]
    a = a[~np.isnan(a)]
    b = values[mask_b]
    b = b[~np.isnan(b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan, np.nan, np.nan

    differences = np.full(n_resamples, np.nan)
    for k in range(n_resamples):
        resample_a = np.random.choice(a, size=len(a), replace=True)
        resample_b = np.random.choice(b, size=len(b), replace=True)
        differences[k] = np.median(resample_b) - np.median(resample_a)

    return (np.median(b) - np.median(a),
            np.percentile(differences, 2.5), np.percentile(differences, 97.5))


def compare_two_groups(values, mask_a, mask_b, name_a="group A", name_b="group B"):
    """Is group A different from group B? Returns a dictionary of numbers.

    Gives you the two medians, the difference with a bootstrap 95% range, and a
    p-value from the Mann-Whitney U test. Computes only; prints nothing.
    """
    a = values[mask_a]
    a = a[~np.isnan(a)]
    b = values[mask_b]
    b = b[~np.isnan(b)]

    if len(a) < 3 or len(b) < 3:
        return {"name_a": name_a, "name_b": name_b, "n_a": len(a), "n_b": len(b),
                "median_a": np.nan, "median_b": np.nan, "difference": np.nan,
                "low": np.nan, "high": np.nan, "p_value": np.nan}

    difference, low, high = compute_difference_ci(values, mask_a, mask_b)
    statistic, p_value = mannwhitneyu(a, b)

    return {"name_a": name_a, "name_b": name_b, "n_a": len(a), "n_b": len(b),
            "median_a": np.median(a), "median_b": np.median(b),
            "difference": difference, "low": low, "high": high, "p_value": p_value}


def format_p(p_value):
    """Write a p-value honestly. A test can never show p is exactly zero."""
    if np.isnan(p_value):
        return "= n/a"
    if p_value < 0.001:
        return "< 0.001"
    return "= %.3f" % p_value


def describe_comparison(result, unit="", indent=2):
    """Print one comparison as a readable sentence."""
    pad = " " * indent
    if np.isnan(result["difference"]):
        print(pad + "not enough trials to compare")
        return

    verdict = "DIFFERENT" if result["p_value"] < 0.05 else "not distinguishable"
    print(pad + "%s: %.1f%s (n=%d)   %s: %.1f%s (n=%d)"
          % (result["name_a"], result["median_a"], unit, result["n_a"],
             result["name_b"], result["median_b"], unit, result["n_b"]))
    # Spell out which way round the subtraction goes, so a minus sign is never
    # ambiguous.
    print(pad + "difference (%s minus %s) %+.1f%s, 95%% range [%+.1f, %+.1f], p %s  -> %s"
          % (result["name_b"], result["name_a"], result["difference"], unit,
             result["low"], result["high"], format_p(result["p_value"]), verdict))


# Worked example: did the reaction time change across the injection?
#
# A rank test compares TWO groups, so three phases means three comparisons:
# before vs during, before vs after, and during vs after.
reaction_time = behaviour["reaction_time"]
valid = valid_for["reaction_time"]

print("Reaction time across the injection (all detected saccades)\n")

for a in range(len(injection_phases)):
    for b in range(a + 1, len(injection_phases)):
        name_a, mask_a = injection_phases[a]
        name_b, mask_b = injection_phases[b]

        result = compare_two_groups(reaction_time,
                                    mask_a & valid, mask_b & valid,
                                    name_a, name_b)
        describe_comparison(result, " ms")
        print()

# Keep these two masks; the interaction example below uses them.
before_mask = injection_phases[0][1]
during_mask = injection_phases[1][1]

### Three things to notice in that output

**A non-significant result can be the informative one.** *Before* versus *after*
comes out not distinguishable — and that is the finding, not a failure: the
reaction time went up during the injection and then came back. Recovery is what
"no difference from baseline" means here. Always say which comparison produced a
null, because "we found nothing" and "it returned to normal" are different claims.

**Three comparisons from three phases.** Every test has a chance of firing by
accident, and running several raises that chance. With p-values of 1e-9 it makes
no practical difference here, but if you compare many measures across many phases
in your report, say how many tests you ran — a reader cannot judge a p-value of
0.04 without knowing it was one of thirty.

**This pools both directions, which waters the effect down.** The drug went into
one hemisphere, so roughly half of these trials should not be affected at all.
Pooled, the before-to-during change is +20 ms; the project's Step 1 splits by
direction and finds +35 ms on one side and +3.5 ms on the other. Deciding what to
pool and what to split is a scientific choice, not a formatting one.

## The interaction test

To ask whether a *change* differs between two groups, we use a **permutation test**.
The logic is worth understanding, because it is the same trick as the bootstrap and
it makes no assumptions at all:

1. Measure the real difference-of-differences — how much more group A changed than
   group B.
2. Now pretend the group labels were meaningless: shuffle them, and measure the
   same thing again.
3. Repeat a couple of thousand times. That gives you the range of values chance
   alone produces.
4. If the real number sits outside that range, the labels mattered.

In [ ]:
def compare_change(values, mask_before, mask_after):
    """Did one group change between two phases? A plain two-group comparison."""
    return compare_two_groups(values, mask_before, mask_after, "before", "after")


def compare_interaction(values, group_a_before, group_a_after,
                        group_b_before, group_b_after, n_permutations=2000):
    """Did group A change MORE than group B did?

    Returns the change in each group, the difference between those changes, and a
    permutation p-value for that difference.
    """
    def median_of(mask):
        chosen = values[mask]
        chosen = chosen[~np.isnan(chosen)]
        return np.median(chosen) if len(chosen) >= 3 else np.nan

    change_a = median_of(group_a_after) - median_of(group_a_before)
    change_b = median_of(group_b_after) - median_of(group_b_before)
    observed = change_a - change_b

    if np.isnan(observed):
        return {"change_a": change_a, "change_b": change_b,
                "difference": np.nan, "p_value": np.nan}

    # Pool the two groups WITHIN each phase, then reshuffle which trials count as
    # group A and which as group B. The phases stay put -- we are testing the
    # group labels, not the phases.
    before_values = np.concatenate([values[group_a_before], values[group_b_before]])
    before_values = before_values[~np.isnan(before_values)]
    after_values = np.concatenate([values[group_a_after], values[group_b_after]])
    after_values = after_values[~np.isnan(after_values)]

    n_a_before = int(np.sum(group_a_before & ~np.isnan(values)))
    n_a_after = int(np.sum(group_a_after & ~np.isnan(values)))

    shuffled = np.full(n_permutations, np.nan)
    for k in range(n_permutations):
        np.random.shuffle(before_values)
        np.random.shuffle(after_values)

        fake_change_a = np.median(after_values[:n_a_after]) - np.median(before_values[:n_a_before])
        fake_change_b = np.median(after_values[n_a_after:]) - np.median(before_values[n_a_before:])
        shuffled[k] = fake_change_a - fake_change_b

    # How often does chance alone produce something at least this big?
    # The +1 on both sides matters: with 2000 shuffles the smallest honest answer
    # is 1/2001, not 0. A permutation test can never prove a p-value of zero, and
    # reporting one would be a false claim.
    n_at_least_as_big = int(np.sum(np.abs(shuffled) >= np.abs(observed)))
    p_value = (n_at_least_as_big + 1) / (n_permutations + 1)

    return {"change_a": change_a, "change_b": change_b,
            "difference": observed, "p_value": p_value}


def describe_interaction(result, name_a="group A", name_b="group B", unit="", indent=2):
    """Print an interaction result as a readable sentence."""
    pad = " " * indent
    if np.isnan(result["difference"]):
        print(pad + "not enough trials to compare")
        return
    verdict = "the two DIFFER" if result["p_value"] < 0.05 else "no detectable difference"
    print(pad + "%s changed by %+.1f%s" % (name_a, result["change_a"], unit))
    print(pad + "%s changed by %+.1f%s" % (name_b, result["change_b"], unit))
    print(pad + "difference between those changes: %+.1f%s, p %s  -> %s"
          % (result["difference"], unit, format_p(result["p_value"]), verdict))


def compare_interaction_anova(values, group_a_before, group_a_after,
                              group_b_before, group_b_after):
    """The same interaction question, asked with a two-way ANOVA instead.

    Builds a table with one row per trial and two factors -- phase (before/after)
    and group (A/B) -- then reads off the interaction term. Returns the F value
    and p-value, or nan if statsmodels is unavailable.
    """
    try:
        import pandas as pd
        import statsmodels.formula.api as smf
        import statsmodels.api as sm_api
    except ImportError:
        print("  statsmodels is not installed -- skipping the ANOVA")
        return {"F": np.nan, "p_value": np.nan}

    rows = []
    for phase_name, group_name, mask in [
            ("before", "A", group_a_before), ("after", "A", group_a_after),
            ("before", "B", group_b_before), ("after", "B", group_b_after)]:
        for value in values[mask]:
            if not np.isnan(value):
                rows.append({"value": value, "phase": phase_name, "group": group_name})

    table = pd.DataFrame(rows)
    model = smf.ols("value ~ C(phase)*C(group)", data=table).fit()
    result = sm_api.stats.anova_lm(model, typ=2)

    interaction = result.loc["C(phase):C(group)"]
    return {"F": interaction["F"], "p_value": interaction["PR(>F)"]}


# Worked example: did RT change MORE for rightward saccades than leftward ones,
# from before the injection to during it?
during_mask = injection_phases[1][1]
valid = valid_for["reaction_time"]

example = compare_interaction(
    behaviour["reaction_time"],
    before_mask & is_right_saccade & valid, during_mask & is_right_saccade & valid,
    before_mask & is_left_saccade & valid, during_mask & is_left_saccade & valid)

print("Change in RT from before to during the injection:")
describe_interaction(example, "rightward", "leftward", " ms")

# The same question, asked the classical way. If the two disagree, find out why
# before choosing one.
anova = compare_interaction_anova(behaviour["reaction_time"],
    before_mask & is_right_saccade & valid, during_mask & is_right_saccade & valid,
    before_mask & is_left_saccade & valid, during_mask & is_left_saccade & valid)
print("\n  two-way ANOVA interaction: F = %.1f, p %s"
      % (anova["F"], format_p(anova["p_value"])))

## One level further: a three-way question

Step 3 of the project needs one more level again. There the question is not "did
these two groups change differently" but:

> Did the **good-minus-bad difference** change across phases **differently on the
> two sides**?

That is three factors at once — phase × side × object value — and the answer lives
in the *three-way* interaction term.

### Fit one model over all the phases, then look inside it

The tempting shortcut is a separate 2 × 2 × 2 ANOVA for each pair of phases. Do not:
that is several independent tests with no protection against one firing by accident,
and each throws away the other phases' trials when estimating how noisy the data are.

The standard approach is better on three counts, and it is what we use:

1. **Fit one model with every phase in it.** The noise estimate is pooled over all
   the trials — here about 1700 rather than 600 — so every comparison rests on a
   steadier estimate of the variance.
2. **Read the omnibus three-way term first.** With four phases it has 3 degrees of
   freedom and asks one yes/no question: *does the value gap behave differently on
   the two sides anywhere across these phases?* It is the gatekeeper. If it is not
   significant, stop.
3. **Then read the contrasts, and correct them.** Only if the gatekeeper passes do
   you ask which phase. Because the model is coded with the baseline as reference,
   the three-way coefficients **are already** those comparisons — one per later
   phase — and they come out in the units of the measure. Holm-correct across them,
   because there is still more than one.

### The contrasts are the interesting part, not the F

An F value says something happened; it does not say how big or which way. Each
three-way coefficient is exactly this, in milliseconds:

```
[ gap(right, this phase) - gap(right, baseline) ]
- [ gap(left,  this phase) - gap(left,  baseline) ]       where gap = good - bad
```

**Order the levels with the reference first** in each list you pass.

### And one more question the three-way term does *not* answer

The three-way term says the value gap moved **differently** on the two sides. It
does not say **which side moved**, and that is a separate contrast — a *simple
effect*. It is entirely possible for the two sides to differ because the control
side moved while the affected side sat still, which would mean something quite
different from what you were hoping to find.

`compare_simple_effects` answers it, reading each side's own value-gap change out
of the same pooled model, so the two tests agree with each other.

In [ ]:
def compare_three_way(values, valid, phase_levels, side_levels, value_levels):
    """Phase x side x object value, as ONE model over all the phases.

    THE DESIGN. Every trial carries three labels: which block of the session it
    came from, which way the saccade went, and whether the object was worth a lot
    or a little. Each argument is that factor's levels, as (name, mask) pairs.

    LIST THE REFERENCE LEVEL FIRST in each list, e.g.
        phase_levels = [("before", ...), ("during", ...), ("after", ...)]
        side_levels  = [("left", ...), ("right", ...)]
        value_levels = [("bad", ...), ("good", ...)]
    so that "gap" means good minus bad and the side contrast means right minus left.

    Returns:
      omnibus_F, omnibus_p   the three-way term over all phases -- the gatekeeper
      contrasts              one entry per non-baseline phase, each with the effect
                             IN THE UNITS OF THE MEASURE, its raw p-value and its
                             Holm-corrected p-value
      table                  the full ANOVA table, if you want to see the rest
    """
    try:
        import pandas as pd
        import statsmodels.formula.api as smf
        import statsmodels.api as sm_api
        from statsmodels.stats.multitest import multipletests
    except ImportError:
        print("  statsmodels is not installed -- skipping")
        return {"omnibus_F": np.nan, "omnibus_p": np.nan, "contrasts": [], "table": None}

    rows = []
    for phase_name, phase_mask in phase_levels:
        for side_name, side_mask in side_levels:
            for value_name, value_mask in value_levels:
                selected = phase_mask & side_mask & value_mask & valid
                for value in values[selected]:
                    if not np.isnan(value):
                        rows.append({"value": value, "phase": phase_name,
                                     "side": side_name, "object": value_name})

    table = pd.DataFrame(rows)

    # The first level of each list becomes the reference that the others are
    # measured against. That is what makes the coefficients the contrasts we want.
    formula = ('value ~ C(phase, Treatment(reference="%s"))'
               ' * C(side, Treatment(reference="%s"))'
               ' * C(object, Treatment(reference="%s"))'
               % (phase_levels[0][0], side_levels[0][0], value_levels[0][0]))

    model = smf.ols(formula, data=table).fit()
    anova = sm_api.stats.anova_lm(model, typ=2)

    # The three-way row is the one naming all three factors.
    three_way_row = [name for name in anova.index if name.count(":") == 2][0]

    # The three-way coefficients: one per non-baseline phase. Each is already the
    # contrast of that phase against the baseline, in the units of the measure.
    coefficient_names = [name for name in model.params.index if name.count(":") == 2]
    raw_p = [model.pvalues[name] for name in coefficient_names]
    _, adjusted_p, _, _ = multipletests(raw_p, method="holm")

    contrasts = []
    for name, adjusted in zip(coefficient_names, adjusted_p):
        phase_name = name.split(":")[0].split("[T.")[1].rstrip("]")
        contrasts.append({"phase": phase_name,
                          "effect": model.params[name],
                          "p_value": model.pvalues[name],
                          "p_adjusted": adjusted})

    # Report them in the order the phases were given, not alphabetically.
    order = [name for name, mask in phase_levels[1:]]
    contrasts.sort(key=lambda c: order.index(c["phase"]) if c["phase"] in order else 99)

    return {"omnibus_F": anova.loc[three_way_row, "F"],
            "omnibus_p": anova.loc[three_way_row, "PR(>F)"],
            "contrasts": contrasts, "table": anova}


def compare_simple_effects(values, valid, phase_levels, side_levels, value_levels):
    """For EACH side on its own: did the good-minus-bad gap change across phases?

    This is a different question from compare_three_way. That one asks whether the
    gap moved DIFFERENTLY on the two sides; this asks whether it moved at all on
    each side taken separately. A "simple effect", in the jargon.

    You need both. A significant three-way term tells you the sides behaved
    differently, but not which side did the moving -- and it is entirely possible
    for the sides to differ because the CONTROL side moved.

    Read from the same pooled model as compare_three_way, so the two agree and the
    noise estimate uses every trial. Returns {side name: [contrasts]}.
    """
    try:
        import pandas as pd
        import statsmodels.formula.api as smf
    except ImportError:
        print("  statsmodels is not installed -- skipping")
        return {}

    rows = []
    for phase_name, phase_mask in phase_levels:
        for side_name, side_mask in side_levels:
            for value_name, value_mask in value_levels:
                selected = phase_mask & side_mask & value_mask & valid
                for value in values[selected]:
                    if not np.isnan(value):
                        rows.append({"value": value, "phase": phase_name,
                                     "side": side_name, "object": value_name})

    table = pd.DataFrame(rows)
    results = {}

    # Fit once per side. Making a side the reference level is what turns the
    # two-way phase:object coefficients into that side's own value-gap change.
    for side_name, side_mask in side_levels:
        formula = ('value ~ C(phase, Treatment(reference="%s"))'
                   ' * C(side, Treatment(reference="%s"))'
                   ' * C(object, Treatment(reference="%s"))'
                   % (phase_levels[0][0], side_name, value_levels[0][0]))
        model = smf.ols(formula, data=table).fit()

        contrasts = []
        for name in model.params.index:
            # A phase:object term with no side in it is the simple effect at the
            # side we made the reference.
            if name.count(":") == 1 and "phase" in name and "object" in name and "side" not in name:
                phase_name = name.split(":")[0].split("[T.")[1].rstrip("]")
                contrasts.append({"phase": phase_name,
                                  "effect": model.params[name],
                                  "p_value": model.pvalues[name]})

        order = [name for name, mask in phase_levels[1:]]
        contrasts.sort(key=lambda c: order.index(c["phase"]) if c["phase"] in order else 99)
        results[side_name] = contrasts

    return results


def describe_simple_effects(results, unit="", baseline_name="baseline"):
    """Print each side's own value-gap change, phase by phase."""
    for side_name, contrasts in results.items():
        print("  %s" % side_name)
        for c in contrasts:
            verdict = "moved" if c["p_value"] < 0.05 else "no detectable change"
            print("     gap change, %-28s %+7.1f%s   p %s  -> %s"
                  % (c["phase"] + " vs " + baseline_name, c["effect"], unit,
                     format_p(c["p_value"]), verdict))
        print()


def describe_three_way(result, unit="", baseline_name="baseline"):
    """Print a three-way result the right way round: the gatekeeper first."""
    if np.isnan(result["omnibus_F"]):
        return

    print("  Omnibus three-way term:  F = %.2f, p %s"
          % (result["omnibus_F"], format_p(result["omnibus_p"])))

    if result["omnibus_p"] >= 0.05:
        print("  Not significant -- stop here. Do not go fishing in the contrasts.")
        return

    print("  Significant, so which phases? (Holm-corrected across the contrasts)")
    for c in result["contrasts"]:
        verdict = "yes" if c["p_adjusted"] < 0.05 else "no"
        print("     %-28s %+7.1f%s   p %s, corrected %s  -> %s"
              % (c["phase"] + " vs " + baseline_name, c["effect"], unit,
                 format_p(c["p_value"]), format_p(c["p_adjusted"]), verdict))

That last number is the shape of every claim worth making in this project: not
"this group changed", but "**this group changed more than that one did**".

One honest caveat to carry into your report. Trials were recorded in order, so
"before" and "during" also differ in *when they happened*. A permutation test tells
you the labels mattered; it cannot tell you the label was the **drug** rather than
the passage of time. The defence against that is the design itself — the drug went
into one hemisphere, so the other direction is a control that experienced exactly
the same passage of time. That is why the interaction, rather than the change, is
the claim to make.

---
---
# 12. The project

## Goal

> **Investigate how reward value shapes the effect of FEF inactivation.**

Silencing the FEF makes saccades to one side worse. The question this project asks
is whether that damage is **the same for everything the monkey looks at**, or
whether a valuable object is partly protected from it.

That is a real question, not an exercise. If the deficit is smaller for high-value
objects, then whatever the FEF does can be partly substituted for by a
value-driven signal from somewhere else. If the deficit is identical, the FEF's
contribution is independent of value. Either answer is worth reporting, and you
cannot tell which it is without doing the work.

## How the project is built

You work up to the goal in three steps, then extend it:

| Step | Question | Why it comes first |
|---|---|---|
| **1** | What does inactivation do at all? | You cannot ask how value changes an effect until you have shown the effect exists |
| **2** | Does value change behaviour? | Same: you need the value effect on its own |
| **3** | **Does inactivation hurt good and bad objects differently?** | The goal. It only means anything with 1 and 2 established |
| **4** | What if value were predictable from location? | An extension to design, not to run |

Sections 8 to 11 built every tool you need. The measures are computed for you; in
this project **you choose what to plot and what to compare.**

## Your toolkit, in one place

```python
# Summarise and plot, split any way you like
rows = compute_summary_by_phase(values, valid, phases, groups)
rows = compute_proportion_by_phase(flag, valid, phases, groups)   # for percentages
plot_summary_by_phase(rows, "y label", "title")
plot_session_course(trial_numbers, values, groups, phases, "y label", "title")
#   ...add method="mean" for firing rates
plot_distributions_by_group(values, valid, groups, "x label", "title")
plot_endpoint_heatmaps(maps, endpoint_edges, target_x, target_y, "title")

# Statistics
compare_two_groups(values, mask_a, mask_b, "name a", "name b")
compare_interaction(values, a_before, a_after, b_before, b_after)
describe_comparison(result, " ms")
describe_interaction(result, "name a", "name b", " ms")
```

`behaviour` holds every per-trial measure and `valid_for` holds the mask each one
may honestly be read on:

`reaction_time`, `peak_velocity`, `duration`, `amplitude`, `radial_error`,
`tangential_error`, `hold_duration`, `anticipated`, `held_to_offset`

### First, two groupings you will reuse

The whole project turns on comparing **four** conditions at once: each direction
crossed with each object value. Building that list once, here, keeps every figure
below to two lines.

In [ ]:
# --- the four-way grouping the project needs -----------------------------------
VALUE_DIRECTION_GROUPS = [
    ("left, good", is_left_saccade & is_good_object),
    ("left, bad", is_left_saccade & is_bad_object),
    ("right, good", is_right_saccade & is_good_object),
    ("right, bad", is_right_saccade & is_bad_object),
]

# The same four, with colours, for the session-course plots. Light/dark pairs so
# direction is the hue and value is the shade.
VALUE_DIRECTION_COURSE = [
    ("Left, good", is_left_saccade & is_good_object & saccades["detected"], "#648FFF"),
    ("Left, bad", is_left_saccade & is_bad_object & saccades["detected"], "#88CCEE"),
    ("Right, good", is_right_saccade & is_good_object & saccades["detected"], "#FE6100"),
    ("Right, bad", is_right_saccade & is_bad_object & saccades["detected"], "#FFB000"),
]

for name, mask in VALUE_DIRECTION_GROUPS:
    print("%-14s %4d trials" % (name, np.sum(mask & saccades["detected"])))

### And a finer set of phases

The "after" phase is by far the longest, and the drug keeps changing inside it — it
washes in, peaks, and starts to wear off. One bar spanning a thousand trials blurs
all of that together. So we split the longest phase in half and get a finer time
axis, without hardcoding which phase is the long one.

In [ ]:
def compute_phase_blocks(phases, trial_numbers, min_trials_to_split=200):
    """Split the LONGEST phase in half, so a long phase is not read as one lump.

    Returns a new list of (name, mask) pairs, so it can be passed anywhere the
    ordinary injection_phases can.
    """
    sizes = [np.sum(mask) for name, mask in phases]
    longest = int(np.argmax(sizes))

    blocks = []
    for k, (name, mask) in enumerate(phases):
        if k == longest and sizes[k] >= min_trials_to_split:
            in_phase = np.where(mask)[0]
            middle = in_phase[len(in_phase) // 2]
            blocks.append((name + ", 1st half", mask & (trial_numbers <= middle)))
            blocks.append((name + ", 2nd half", mask & (trial_numbers > middle)))
        else:
            blocks.append((name, mask))

    return blocks


phase_blocks = compute_phase_blocks(injection_phases, trial_numbers)

print("Finer phases:")
for name, mask in phase_blocks:
    print("   %-20s %4d trials" % (name, np.sum(mask)))

---
## Step 1 — What does the inactivation do at all?

Before asking how reward interacts with the deficit, establish the deficit. Two
halves: the neuron, and the behaviour.

### 1.1 The neuron

Muscimol silences neurons. Show what happened to the one you recorded.

**Produce:**

1. The neuron's firing rate **across the session**, so you can see the drug wash in.
2. Its PSTH in each injection phase, so you can see *which part* of the response
   changed — the baseline, the visual response, or both.
3. A bar chart of firing rate by phase, with error bars.

A caution you must handle: **a firing rate is meaningless unless you say what it
was aligned to.** The same −200 to 0 ms window is genuine rest before the *target*,
but sits in the middle of the delay period before the *GO cue*. On this session
those give 5.2 and 15.7 spikes/s. State which you used.

In [ ]:
# --- 1.1 the neuron ------------------------------------------------------------
# Worked for you: the firing rate across the session.
#
# Note method="mean". A rate measured in a 200 ms window can only be 0, 5, 10 ...
# spikes/s, because you cannot have half a spike, so a running MEDIAN of those
# snaps to those few values and draws a staircase. A running mean reads properly.
# Try changing it to "median" once, to see what the difference looks like.
plot_session_course(trial_numbers, visual_rate,
                    [("All trials", ALL_TRIALS, "tab:purple")],
                    injection_phases, "Firing rate (spikes/s)",
                    "1.1  Response to the target (50-250 ms) across the session",
                    method="mean")

# TODO 1: draw the same thing for baseline_rate. Does the baseline fall as much as
#         the visual response does, or is one affected more than the other?
# TODO 2: draw one PSTH per injection phase (compute_psth with a phase mask, as in
#         section 10) on a single axis, so you can see WHICH part of the response
#         changed rather than only that something did.
# TODO 3: make the bar chart:
#             rows = compute_summary_by_phase(visual_rate, ALL_TRIALS,
#                                             injection_phases, DIRECTION_GROUPS)
#             plot_summary_by_phase(rows, "Firing rate (spikes/s)", "...")
#         Does the neuron respond more to one direction, and does the drug change
#         that preference or just scale it down?

### 1.2 The behaviour

Now the same story on the behavioural side. **Plot each measure against trial
number**, so the wash-in and wash-out are visible, then summarise it as bars with
error bars, then put a statistic on it.

**Produce:** for `reaction_time`, `peak_velocity`, `duration` and `radial_error` —

1. a session course, split by direction
2. a bar chart by phase and direction, with error bars
3. one statistical statement

Remember what each measure is for. RT says **when** the movement started; peak
velocity says **how fast** it went; duration and amplitude together say whether a
slow saccade really covered the same ground; radial error says whether it **fell
short**. They can move independently, and that is the interesting part.

Below, `reaction_time` is worked through completely as a template. Do the other
three yourself.

In [ ]:
# --- 1.2 behaviour: reaction time, worked as a template ------------------------
measure = "reaction_time"
values = behaviour[measure]
valid = valid_for[measure]

# (a) across the session
plot_session_course(trial_numbers, values, direction_groups, injection_phases,
                    "Reaction time (ms)", "1.2  Reaction time across the session")

# (b) summarised by phase, with error bars
rows = compute_summary_by_phase(values, valid, phase_blocks, DIRECTION_GROUPS)
plot_summary_by_phase(rows, "Median reaction time (ms)",
                      "1.2  Reaction time by phase and direction")

# (c) and a statistic: did each direction change from before to during?
before_mask = injection_phases[0][1]
during_mask = injection_phases[1][1]

print("Reaction time, before vs during the injection:")
for group_name, group_mask in DIRECTION_GROUPS:
    result = compare_two_groups(values,
                                before_mask & group_mask & valid,
                                during_mask & group_mask & valid,
                                "before", "during")
    print(" %s" % group_name)
    describe_comparison(result, " ms")

# ...and the claim that actually matters: did they change DIFFERENTLY?
print("\nDid the two directions change differently?")
interaction = compare_interaction(values,
    before_mask & is_right_saccade & valid, during_mask & is_right_saccade & valid,
    before_mask & is_left_saccade & valid, during_mask & is_left_saccade & valid)
describe_interaction(interaction, "rightward", "leftward", " ms")

In [ ]:
# --- 1.2 YOUR TURN -------------------------------------------------------------
# Repeat the three steps above for each of these. The code is identical apart from
# the measure name, so a loop will do it -- but read each figure before moving on.

for measure in ["peak_velocity", "duration", "radial_error"]:
    values = behaviour[measure]
    valid = valid_for[measure]

    # TODO 1: a session course for this measure (copy line (a) above)
    # TODO 2: a bar chart by phase and direction (copy line (b) above)
    # TODO 3: the interaction test (copy the last block above)
    pass

# TODO 4: peak velocity and RT do NOT have the same time course on this session.
#         One recovers and one does not. Which is which, and what would that mean
#         about the two things the FEF contributes?
# TODO 5: if peak velocity drops while amplitude holds, duration MUST rise. Check
#         that it does. If it did not, you would have an internal contradiction
#         and should suspect the detector before believing the result.
# TODO 6: radial_error is negative when the eye falls short. Does the affected
#         side undershoot more after the injection?
# TODO 7: endpoint SCATTER is separate from endpoint ERROR -- accuracy versus
#         precision. Use compute_endpoint_scatter for each phase and direction.
#         Did the saccades get more variable, less accurate, or both?

### 1.2b The endpoints, in two dimensions

`radial_error` told you *how far* the eye missed along the axis to the target.
The heatmap from section 9 shows the whole landing distribution at once, and it is
worth drawing across the finer blocks: accuracy and precision can drift within the
long "after" phase, and one map per phase would average that away.

In [ ]:
# --- 1.2b endpoints as 2-D maps, block by block --------------------------------
maps = compute_endpoint_heatmap(saccades["end_x"], saccades["end_y"],
                                phase_blocks, endpoint_edges)

plot_endpoint_heatmaps(maps, endpoint_edges, target_x, target_y,
                       "1.2b  Landing points across the session "
                       "(+ = fixation, o = targets)")

# TODO 1: does either blob move TOWARDS the fixation cross in the later panels?
#         That is hypometria, and it should appear on one side only.
# TODO 2: does either blob get more spread out? Put a number on it with
#         compute_endpoint_scatter, per block and per direction -- the eye can
#         become imprecise without becoming inaccurate.
# TODO 3: split the maps by object value instead of by direction, by passing
#         groups built from is_good_object and is_bad_object. Does the monkey aim
#         more carefully at the valuable object? (That is a Step 2 question, but
#         the map is the clearest way to look at it.)

### 1.3 Is there compensation while the FEF is inactivated?

The last question of Step 1, and it is about the inactivation alone — no reward,
no object value yet.

A deficit is not necessarily a fixed thing. Over hundreds of trials with a
half-silenced FEF, the animal may **adapt**: recruiting other structures,
re-calibrating, or simply trying harder. If so, the deficit should shrink as the
inactivation phase goes on.

**Measure the deficit, not the raw reaction time.** Both directions get faster late
in this session — the monkey anticipates the go cue more and more — so a falling
rightward RT does not by itself mean recovery. The **difference between the
directions** cancels that out, because anticipation affects both. This is the
single most important choice in this sub-step.

### The confound you cannot dodge, and the one handle you have on it

A shrinking deficit has two explanations, and they look identical in the behaviour:

1. **The drug wore off.** Muscimol washes out over tens of minutes.
2. **The monkey compensated.** The drug is still acting; the behaviour adapted.

Nothing in the behaviour alone separates those. But **you recorded a neuron**, and
that is your handle:

- if the neuron's firing recovers *in step* with the behaviour → washout
- if the behaviour recovers *while the neuron is still suppressed* → compensation
- if the neuron recovers but the behaviour does not → the deficit outlived the
  silencing, which points at something downstream

Plot the two time courses together and compare their shapes. That comparison is
the whole sub-step.

In [ ]:
# --- 1.3 compensation ----------------------------------------------------------
def compute_deficit_by_block(values, valid, blocks, affected_mask, unaffected_mask):
    """The size of the deficit in each block: affected minus unaffected side.

    Using the DIFFERENCE between the two directions cancels anything that moved
    both of them together, such as the monkey anticipating more as he warms up.
    """
    names, deficits, lows, highs = [], [], [], []

    for block_name, block_mask in blocks:
        affected = block_mask & affected_mask & valid
        unaffected = block_mask & unaffected_mask & valid
        difference, low, high = compute_difference_ci(values, unaffected, affected)

        names.append(block_name)
        deficits.append(difference)
        lows.append(low)
        highs.append(high)

    return names, np.array(deficits), np.array(lows), np.array(highs)


def plot_deficit_and_neuron(names, deficits, lows, highs, neuron_rate,
                            blocks, y_label, title):
    """The behavioural deficit and the neuron's firing rate, side by side in time.

    Two different units, so two y axes -- the point is to compare the SHAPES of
    the two curves, not their heights.
    """
    fig, ax = plt.subplots(figsize=(10, 4.8))
    positions = np.arange(len(names))

    ax.errorbar(positions, deficits, yerr=[deficits - lows, highs - deficits],
                fmt="o-", capsize=6, linewidth=2, markersize=9,
                color="tab:red", label="behavioural deficit")
    ax.axhline(0, color="black", linestyle="--", linewidth=1.2)
    ax.set_ylabel(y_label, color="tab:red")
    ax.tick_params(axis="y", labelcolor="tab:red")

    # The neuron on its own axis.
    other = ax.twinx()
    rates = [np.nanmean(neuron_rate[block_mask]) for block_name, block_mask in blocks]
    other.plot(positions, rates, "s--", color="tab:purple", linewidth=2,
               markersize=8, label="neuron firing rate")
    other.set_ylabel("Firing rate (spikes/s)", color="tab:purple")
    other.tick_params(axis="y", labelcolor="tab:purple")

    ax.set_xticks(positions)
    ax.set_xticklabels(names, rotation=20, ha="right")
    ax.set_title(title)

    # One legend for both axes.
    lines_a, labels_a = ax.get_legend_handles_labels()
    lines_b, labels_b = other.get_legend_handles_labels()
    ax.legend(lines_a + lines_b, labels_a + labels_b, loc="best")

    fig.tight_layout()
    plt.show()


names, deficits, lows, highs = compute_deficit_by_block(
    behaviour["reaction_time"], valid_for["reaction_time"], phase_blocks,
    is_right_saccade, is_left_saccade)

plot_deficit_and_neuron(names, deficits, lows, highs, visual_rate, phase_blocks,
                        "RT deficit: rightward minus leftward (ms)",
                        "1.3  Does the deficit shrink while the neuron is still suppressed?")

print("Deficit by block (rightward minus leftward RT):")
for name, deficit, low, high in zip(names, deficits, lows, highs):
    print("   %-20s %+6.0f ms  [%+.0f, %+.0f]" % (name, deficit, low, high))

### The same question for every measure at once

One measure can mislead. RT might recover while peak velocity does not, and a
single figure showing all of them across the same blocks is the fastest way to see
that. This is the summary figure for Step 1.

In [ ]:
def plot_measure_panel(measure_names, behaviour, valid_for, blocks, groups,
                       units, title):
    """One small panel per measure, each showing every group across the blocks.

    Lets you compare the SHAPE of the time course between measures that have
    completely different units.
    """
    n_measures = len(measure_names)
    n_columns = min(3, n_measures)
    n_rows = int(np.ceil(n_measures / n_columns))

    fig, axes = plt.subplots(n_rows, n_columns,
                             figsize=(4.6 * n_columns, 3.6 * n_rows), squeeze=False)

    block_names = [name for name, mask in blocks]
    colours = ["tab:blue", "tab:red", "tab:green", "tab:orange"]

    for k, measure_name in enumerate(measure_names):
        ax = axes[k // n_columns][k % n_columns]
        values = behaviour[measure_name]
        valid = valid_for[measure_name]

        for g, (group_name, group_mask) in enumerate(groups):
            middles, downs, ups = [], [], []
            for block_name, block_mask in blocks:
                selected = block_mask & group_mask & valid
                chosen = values[selected]
                chosen = chosen[~np.isnan(chosen)]
                if len(chosen) < 3:
                    middles.append(np.nan); downs.append(0); ups.append(0)
                    continue
                middle = np.median(chosen)
                low, high = compute_bootstrap_ci(chosen)
                middles.append(middle)
                downs.append(middle - low)
                ups.append(high - middle)

            ax.errorbar(np.arange(len(blocks)), middles, yerr=[downs, ups],
                        fmt="o-", capsize=4, linewidth=2, markersize=7,
                        color=colours[g % len(colours)], label=group_name)

        ax.set_title("%s (%s)" % (measure_name.replace("_", " "),
                                  units.get(measure_name, "")), fontsize=10)
        ax.set_xticks(np.arange(len(blocks)))
        ax.set_xticklabels(block_names, rotation=30, ha="right", fontsize=8)
        if k == 0:
            ax.legend(fontsize=8)

    for k in range(n_measures, n_rows * n_columns):
        axes[k // n_columns][k % n_columns].axis("off")

    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    plt.show()


plot_measure_panel(["reaction_time", "peak_velocity", "duration",
                    "amplitude", "radial_error", "hold_duration"],
                   behaviour, valid_for, phase_blocks, DIRECTION_GROUPS, UNITS,
                   "1.3  Every measure across the session -- "
                   "does the gap between the two directions close?")

**How to read it for compensation.** In each panel, look at the *vertical gap*
between the two lines, not at their heights. That gap is the deficit. If it is
widest during the injection and narrows afterwards, something recovered — and the
neuron panel from the previous figure tells you whether the recovery was the drug
washing out or the animal adapting.

Watch for the two lines moving *together*, up or down, across the blocks. That is
not compensation; it is something affecting both directions, and on this session
it is largely the monkey anticipating the go cue more as he goes on.

In [ ]:
# --- 1.3 YOUR TURN -------------------------------------------------------------
# TODO 1: the four blocks above are coarse. Split the longest phase into four or
#         six instead, by writing your own list of (name, mask) pairs the way
#         compute_phase_blocks does, and see whether the deficit really declines
#         or just bounces around. With this many trials per block, is the
#         apparent trend bigger than the error bars?
#
# TODO 2: compare the two curves' SHAPES. Does the neuron recover smoothly while
#         the deficit stays put, or do they move together? Write down which of
#         the three interpretations above your figure supports.
#
# TODO 3: do the same for peak_velocity. On this session RT and peak velocity do
#         not recover on the same schedule, so "did the animal compensate?" may
#         have different answers for initiation and for execution -- which would
#         be a much more interesting result than a single yes or no.
#
# TODO 4: the trap named above, checked directly. Plot the raw median RT for each
#         direction across the blocks, next to the deficit. Both directions get
#         faster late in the session. If you had used rightward RT alone, what
#         would you have concluded, and why would it have been wrong?
#
# TODO 5: check the anticipation rate across the same blocks
#         (compute_proportion_by_phase with behaviour["anticipated"]). If it
#         climbs, that is the thing making both directions faster.
#
# TODO 6: endpoint accuracy has its own compensation question. Using the panel
#         figure and the 1.2b heatmaps, does the landing accuracy recover on the
#         same schedule as the reaction time? Add endpoint SCATTER per block with
#         compute_endpoint_scatter -- it is not in the panel figure, because it
#         is a property of a group rather than of a trial and so has no error bar.
#
# TODO 7: write one sentence per measure saying whether it compensated. Then say
#         which single measure you would give as the answer, and why.

---
## Step 2 — Does reward value change the behaviour?

Now the other ingredient, **on its own**. This step uses only the trials from
**before the injection**, where the brain was intact. The point is simply to find
out what reward value does to this monkey's behaviour normally.

> Everything about how the injection and reward value interact is Step 3. Do not
> bring the injection phases in here — you cannot interpret a change in the value
> effect until you know what the value effect *is*.

Reward could show up in any of these, and each says something different:

| Measure | What a value effect there would mean |
|---|---|
| `reaction_time` | the monkey initiates faster for something worth more |
| `anticipated` | he is *predicting* the go cue more for valuable objects |
| `peak_velocity` | the movement itself is more vigorous |
| `radial_error` | he aims more carefully |
| `hold_duration` | he keeps looking at it once he gets there |
| `held_to_offset` | ...still looking when it disappears |

`hold_duration` is worth your attention. It is the one measure the task does not
constrain — it enforces only a minimum hold — so it is the freest behaviour in the
dataset, and the most likely place for value to appear.

### Restricting to the baseline

There is no new machinery for this. `compute_summary_by_phase` takes any two lists
of `(name, mask)` pairs, so we hand it **direction** as the first factor and
**object value** as the second, and restrict everything to the baseline by adding
the before-injection mask to `valid`.

Splitting by direction as well is worth doing even though both sides are intact
here: if the value effect already differs between the two sides *before* any drug,
that is something Step 3 must take into account rather than discover later.

In [ ]:
# --- 2 worked example: does value change how long he looks? --------------------
baseline = injection_phases[0][1]          # the before-injection trials

measure = "hold_duration"
values = behaviour[measure]
valid = valid_for[measure] & baseline      # <- the whole restriction, in one place

rows = compute_summary_by_phase(values, valid, DIRECTION_GROUPS, VALUE_GROUPS)
plot_summary_by_phase(rows, "Gaze hold on target (ms)",
                      "2  How long the monkey kept looking, before the injection")

plot_distributions_by_group(values, valid, VALUE_GROUPS,
                            "Gaze hold on target (ms)",
                            "2  The full distributions, before the injection")

print("Gaze hold, good versus bad object (before the injection):")
result = compare_two_groups(values,
                            baseline & is_good_object & valid,
                            baseline & is_bad_object & valid,
                            "good object", "bad object")
describe_comparison(result, " ms")

In [ ]:
# --- 2 YOUR TURN ---------------------------------------------------------------
# Everything below stays inside the baseline trials. Just change `measure`.

# TODO 1: run the same three blocks for reaction_time, peak_velocity and
#         radial_error. Which measures carry a value effect and which do not?
#         A measure with NO value effect is a useful result, not a failed one --
#         and it matters for Step 3, because a measure with no value effect at
#         baseline cannot show the injection changing that effect.
#
# TODO 2: anticipated and held_to_offset are percentages, so they need
#         compute_proportion_by_phase instead:
#             rows = compute_proportion_by_phase(behaviour["anticipated"],
#                                                valid_for["anticipated"] & baseline,
#                                                DIRECTION_GROUPS, VALUE_GROUPS)
#             plot_summary_by_phase(rows, "Anticipated (%)", "...")
#         Does he jump the gun more for the valuable object?
#
# TODO 3: careful with that anticipation number. Only CORRECT trials are in this
#         file, and the trials where he anticipated hardest became fixation breaks
#         and were deleted. Every anticipation rate here is an underestimate. Is it
#         evenly biased across good and bad? If not, what does that do to the
#         comparison?
#
# TODO 4: the good and bad objects were not evenly spread across the two
#         directions. Check whether your clearest value effect holds within EACH
#         direction separately -- the bars are already split that way, so read
#         them, and run compare_two_groups inside one direction to confirm.
#
# TODO 5: write down, in one line per measure, the size of the value effect at
#         baseline. That list is what Step 3 will ask whether the injection
#         changed, so you need it before going on.

---
## Step 3 — The goal: does inactivation hurt good and bad objects differently?

You have a deficit (Step 1) and a value effect (Step 2). Now put them together.

### The quantity to track is the value GAP, not the raw measure

Comparing good against bad within one condition is not enough, because everything
drifts over a long session. The thing to follow is the **good-minus-bad gap**, and
the question is how that gap behaves on the two sides:

> Does the good-minus-bad gap on the **affected** side change across phases
> **differently** from the gap on the **unaffected** side?

Written out, the quantity is a difference of differences of differences:

```
gap(side, phase)  =  median(good) - median(bad)          for that side and phase

what we test      =  [ gap(affected, later) - gap(affected, before) ]
                   - [ gap(intact,   later) - gap(intact,   before) ]
```

That is the **three-way interaction**: phase × side × object value. The unaffected
side is inside the measurement rather than bolted on afterwards, which is exactly
what you want — anything that changes the value gap over a session for reasons
having nothing to do with the drug (the monkey getting less thirsty, say, and so
caring less about the difference between a big and a small reward) moves *both*
sides and cancels here.

### Keep the sign

Use the signed gap, not its absolute value. `|good − bad|` would hide a gap that
**flips sign** — and on this session one of the two sides does exactly that. A sign
flip is a real finding; do not average it away.

### Three sub-steps

1. **Look at the gaps.** Plot them per side, per block, with error bars.
2. **Test it.** Three-way ANOVA on the phase pair you care about.
3. **Decompose it.** If the gap moved, find out *which* condition moved — did the
   good-object trials get worse, or the bad-object ones get better?

Step 3.3 is the part most people skip, and it is where the interpretation lives. A
gap can widen because the top went up or because the bottom came down, and those
are different claims about what the FEF was contributing.

In [ ]:
# --- 3.1 the value gap on each side, block by block ----------------------------
def compute_value_gap(values, valid, blocks, side_mask):
    """The good-minus-bad difference in each block, with a bootstrap range."""
    names, gaps, lows, highs = [], [], [], []

    for block_name, block_mask in blocks:
        good = block_mask & side_mask & is_good_object & valid
        bad = block_mask & side_mask & is_bad_object & valid
        difference, low, high = compute_difference_ci(values, bad, good)

        names.append(block_name)
        gaps.append(difference)
        lows.append(low)
        highs.append(high)

    return names, np.array(gaps), np.array(lows), np.array(highs)


def plot_value_gaps(values, valid, blocks, y_label, title):
    """The good-minus-bad gap on each side, and the difference between the two."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

    stored = {}
    for side_name, side_mask, colour in [("rightward (affected)", is_right_saccade, "tab:red"),
                                         ("leftward (intact)", is_left_saccade, "tab:blue")]:
        names, gaps, lows, highs = compute_value_gap(values, valid, blocks, side_mask)
        stored[side_name] = gaps
        positions = np.arange(len(names))
        axes[0].errorbar(positions, gaps, yerr=[gaps - lows, highs - gaps],
                         fmt="o-", capsize=5, linewidth=2, markersize=8,
                         color=colour, label=side_name)

    axes[0].axhline(0, color="black", linestyle="--", linewidth=1.5)
    axes[0].set_ylabel("Good minus bad, " + y_label)
    axes[0].set_title("The value gap on each side")
    axes[0].legend(fontsize=9)

    # The difference BETWEEN the two gaps -- the quantity the test is about.
    difference = stored["rightward (affected)"] - stored["leftward (intact)"]
    axes[1].plot(np.arange(len(names)), difference, "o-", color="tab:purple",
                 linewidth=2.5, markersize=9)
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1.5)
    axes[1].set_ylabel("Affected gap minus intact gap (" + y_label + ")")
    axes[1].set_title("The difference between them\n(this is what the test asks about)")

    for ax in axes:
        ax.set_xticks(np.arange(len(names)))
        ax.set_xticklabels(names, rotation=20, ha="right")

    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    plt.show()

    return names, difference


names, gap_difference = plot_value_gaps(
    behaviour["reaction_time"], valid_for["reaction_time"], phase_blocks,
    "median RT (ms)", "3.1  How the reward-value gap evolves on each side")

print("Affected-minus-intact value gap, block by block:")
for name, value in zip(names, gap_difference):
    print("   %-20s %+6.0f ms" % (name, value))

In [ ]:
# --- 3.2 the formal test -------------------------------------------------------
# The three factors, one list of levels each. Every trial carries all three
# labels: which block it came from, which way the saccade went, and whether the
# object was worth a lot or a little.
#
# Note the ORDER inside each list: the first level is the reference, so "gap"
# below means good minus bad, and the side contrast means right minus left.
phases_for_test = phase_blocks                                   # baseline first
sides = [("left", is_left_saccade), ("right", is_right_saccade)]
objects = [("bad", is_bad_object), ("good", is_good_object)]

result = compare_three_way(behaviour["reaction_time"], valid_for["reaction_time"],
                           phases_for_test, sides, objects)

print("Three-way test on reaction time: phase x side x object value")
print("(one model over all %d blocks, %s as the baseline)\n"
      % (len(phases_for_test), phases_for_test[0][0]))

describe_three_way(result, " ms", baseline_name=phases_for_test[0][0])

print()
print("Each contrast is the change in the BETWEEN-SIDE value gap, in ms:")
print("   [gap(right, phase) - gap(right, before)] - [gap(left, phase) - gap(left, before)]")
print("A positive number means the value gap grew more on the affected side.")

### 3.3 Which side, and which object, produced that change?

A significant three-way term says the value gap moved differently on the two sides.
It does not say **how**. Four different things could produce the same number:

- on the affected side, the good-object trials got worse
- on the affected side, the bad-object trials got better
- the same, on the intact side, in the opposite direction
- some combination of those

These are different claims about what the FEF was contributing, so the next step is
to take the effect apart. Two questions, in order:

1. **Which side's gap moved?** Test the value gap change within each side on its
   own — a two-way interaction (phase × object value), run once per side.
2. **Within that side, which object moved?** Test the good trials and the bad
   trials separately, before against later.

The figure and the statistics below do exactly that, in that order.

In [ ]:
# --- 3.3 decompose: which condition actually moved? ----------------------------
# A gap can widen because the good trials got worse or because the bad trials got
# better. Those are different claims, so look at all four conditions separately.
rows = compute_summary_by_phase(behaviour["reaction_time"], valid_for["reaction_time"],
                                phase_blocks, VALUE_DIRECTION_GROUPS)

plot_summary_by_phase(rows, "Median reaction time (ms)",
                      "3.3  All four conditions -- which one moved?")

# The same four as lines, which makes the shapes easier to compare than bars.
fig, ax = plt.subplots(figsize=(9.5, 4.8))
colours = ["#648FFF", "#88CCEE", "#FE6100", "#FFB000"]

for (group_name, group_mask), colour in zip(VALUE_DIRECTION_GROUPS, colours):
    medians = []
    for block_name, block_mask in phase_blocks:
        selected = block_mask & group_mask & valid_for["reaction_time"]
        medians.append(np.nanmedian(behaviour["reaction_time"][selected]))
    ax.plot(range(len(phase_blocks)), medians, "o-", color=colour,
            linewidth=2, markersize=8, label=group_name)

ax.set_xticks(range(len(phase_blocks)))
ax.set_xticklabels([name for name, mask in phase_blocks], rotation=20, ha="right")
ax.set_ylabel("Median reaction time (ms)")
ax.set_title("3.3  The four conditions over time")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# --- 3.3 the statistics behind those figures -----------------------------------
measure = "reaction_time"
values = behaviour[measure]
valid = valid_for[measure]

# The block the three-way test flagged. Change this to look at a different one.
later_name, later_mask = phase_blocks[-1]
baseline_name, baseline_mask = phase_blocks[0]

print("STEP 1 -- which side's value gap actually moved?\n")

# Read from the SAME pooled model as the three-way test above, so the two
# tests agree with each other and share one estimate of the noise.
simple = compare_simple_effects(values, valid, phases_for_test, sides, objects)
describe_simple_effects(simple, " ms", baseline_name=phase_blocks[0][0])

print("\nSTEP 2 -- within each side, which object actually moved?\n")

for side_name, side_mask in [("rightward (affected)", is_right_saccade),
                             ("leftward (intact)", is_left_saccade)]:
    print("  %s" % side_name)
    for object_name, object_mask in [("good object", is_good_object),
                                     ("bad object", is_bad_object)]:
        result = compare_two_groups(values,
                                    baseline_mask & side_mask & object_mask & valid,
                                    later_mask & side_mask & object_mask & valid,
                                    baseline_name, later_name)
        print("    %s" % object_name)
        describe_comparison(result, " ms", indent=6)
    print()

In [ ]:
# --- 3 YOUR TURN ---------------------------------------------------------------
# Everything above was done for reaction_time. Now repeat it for the other
# behavioural measures. The code is identical apart from the measure name, so the
# work is in reading the figures, not in writing more code.
#
# For each measure: run 3.1 (the gap figure), 3.2 (the three-way test) and 3.3
# (the decomposition figure and statistics).

for measure in ["peak_velocity", "duration", "radial_error", "hold_duration"]:
    values = behaviour[measure]
    valid = valid_for[measure]

    # TODO 1: the 3.1 figure --
    #     plot_value_gaps(values, valid, phase_blocks, "median %s" % measure, "...")
    # TODO 2: the 3.2 test --
    #     result = compare_three_way(values, valid, phases_for_test, sides, objects)
    #     describe_three_way(result, UNITS[measure], phase_blocks[0][0])
    # TODO 3: the 3.3 decomposition, by copying the two blocks above.
    pass

# TODO 4: peak_velocity is the one to look at hardest. Unlike reaction time, it
#         was still depressed in the "after" blocks -- so if the value effect has
#         anything to do with the inactivation, this is the measure where it
#         should show up. Does it?
#
# TODO 5: hold_duration needs valid_for["hold_duration"], which drops the censored
#         trials. It also had the largest value effect at baseline in Step 2, so it
#         has the most room to change.
#
# TODO 6: collect your answers into one table -- one row per measure, with the
#         three-way omnibus p, the contrast for the last block in its own units,
#         and which condition moved. That table is the answer to the project.
#
# TODO 7: look at result["table"] for the whole ANOVA, not just the three-way row.
#         C(side):C(object) says whether the value gap differed between the sides
#         to begin with, which Step 2 showed it did. An effect that was already
#         there before the injection cannot have been caused by it.

---
## Step 4 — Extension: what if value were predictable from location?

**This step is a design exercise. There is no data for it — you argue it.**

In this task the good and bad objects appeared **randomly** on either side, so the
monkey could not know which he would get until the object appeared. Now imagine the
experiment changed so that the good object **always** appeared in one hemifield and
the bad object always in the other.

Answer these using what you measured above as evidence:

1. **What would change, and why?** The monkey could now prepare before the object
   appeared. Which of your measures would move first — reaction time, anticipation
   rate, or gaze hold? Your Step 2 results say which measures carry value at all,
   and only those can carry a *prediction* of value.

2. **Would compensation be FASTER under that design?** You already measured
   whether the deficit shrinks on its own, in Step 1.3. The question here is
   whether *predictability* would speed that up: if value can be read off the
   location, the monkey could begin preparing before the object appears, and a
   prepared movement may need less from the FEF than one assembled on the fly.
   Use your Step 1.3 answer as the baseline that this design would have to beat.

3. **Or would it not help at all?** If the FEF is required for the *motor output*
   rather than the *decision*, no amount of preparation helps — the signal still
   has to get out through damaged tissue. Your Step 1 result is the evidence:
   **did peak velocity recover, or did it stay down?** A deficit that does not
   recover on its own is unlikely to be fixed by prediction.

4. **How would you tell those apart?** Design the control. What would you have to
   measure, and what result would distinguish "he compensated" from "the drug wore
   off"? Note that you have a hint already: one of the two sessions in this dataset
   is a One-Direction-Rewarded task, where reward *is* tied to direction.

5. **What is the confound?** If good objects are always on the affected side, then
   "good object" and "affected side" are no longer separable. How would you design
   around that? (Hint: what would you have to do across sessions?)